# [실습] 168시간 인수인계표 — 확신과 보류 구분

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

## 0. 오늘 쓸 데이터와 오늘 쓸 차수

<div style="background:#FFFFFF;color:#000000;padding:40px;border-radius:12px;border:1px solid #E6E6E6;border-left:4px solid #051C2C">
<div style="font-size:0.9em;color:#656565">시계열 예측 · 셋째 날</div>
<h1 style="color:#000000;margin:10px 0">SARIMA 로 168시간 채우기</h1>
<div style="font-size:1.05em;color:#656565">어제 고른 차수에 계절 항을 더해 다음 7일을 한 번에 예측하고, 그 표에 시각마다 등급을 적어 인수인계표로 만드는 순서입니다</div>
</div>

## 오늘 할 일

배차팀장이 어제 요청한 표는 아직 비어 있습니다. **"다음 주 트럭 배차표를 금요일까지 결정해야 합니다. 요일별이 아니라 시간대별로, 다음 7일 168시간의 수요를 숫자로 주세요."**

질문은 하나입니다. **SARIMA 로 채운 168시간을 배차팀에 그대로 전달해도 되는가.**

- **95% 예측구간**: 실측이 95% 확률로 들어올 범위입니다. 11월 17일 0시라면 점 예측 625대를 가운데 둔 241～1,008대입니다.
- **등급**: 배차 대수를 금요일에 확정할 **확신**과 당일 아침에 다시 정할 **보류**로 시각을 나눈 열입니다.
- **나의 역할**: 미션 1 이 코어이고, 과제는 이어서 하거나 나중에 해도 됩니다. 코드는 AI 튜터가 쓰고, 나는 시킬 일을 정하고 받은 결과를 확인합니다.

<img src="https://otexts.com/fpp3/fpp_files/figure-html/auscafecombPI-1.png" width="640" height="320"/>
<p align="center"><em>▲ 호주 월별 외식 지출액에 대해 만든 예측. 검은 선이 실측, 파란 선이 예측이고 짙은 영역이 80% · 옅은 영역이 95% 예측구간이다 (출처: Hyndman &amp; Athanasopoulos, Forecasting: Principles and Practice 3rd ed., otexts.com/fpp3)</em></p>

## 오늘의 데이터

**Seoul Bike Sharing Demand** 는 UCI 머신러닝 저장소가 공개한 서울 공유자전거의 시간당 대여 기록입니다. 2017년 12월 1일 0시부터 8,760시간이 한 행씩(14열) 빠짐없이 있습니다.

| 열 | 무엇 | 쓰는 곳 |
|---|---|---|
| `functioning` | 운영 여부. `"No"` 인 **휴무 295시간**은 시스템이 멈춰 대여가 0으로 적힌 시각이고, 이 0 은 수요가 아니라 관측이 없다는 뜻 | 휴무 표시 |
| `count`(원본 시간 계열) | 시간당 대여 수. 휴무 시각은 0 | 탐색과 검정 |
| `demand`(보정 수요) | 휴무 시각을 1주일 전 같은 시각 값으로 채운 계열. 「준비 셀」이 만든다 | 모형 학습·예측·평가 |

<img src="https://commons.wikimedia.org/wiki/Special:FilePath/Bicycle-sharing%20station%20at%20Hangang%20bikeway%202.jpg?width=1600" width="640" height="467"/>
<p align="center"><em>▲ 거치대에 정렬된 공유자전거. 오늘 넘길 168시간 표로 정하는 것은 「몇 대를 몇 시에 채워 두는가」다 (출처: Wikimedia Commons, Bicycle-sharing station at Hangang bikeway)</em></p>

## SARIMA 는 어제 모형에 무엇을 더하나

어제 모형과 오늘 SARIMA 는 둘 다 계절차분한 값 $z$(지금 값 − 24시간 전 같은 시각 값)를 예측하고, 예측에 쓰는 값만 다릅니다.

| 차수 | 1시간 전 $z$·잔차 | 24시간 전 같은 시각 $z$·잔차 |
|---|---|---|
| 어제 (1, 0, 1)(0, 1, 0)₂₄ | 씀 | 쓰지 않음 |
| 오늘 (1, 0, 1)(1, 1, 1)₂₄ | 씀 | 씀 |

같은 학습 1,848시간에서 계절 항 2개를 더하자 AIC(작을수록 나은 점수)가 706.65 줄었습니다. 「준비 셀」이 출력하는 계수 4개 $\phi_1$ · $\theta_1$ · $\Phi_1$ · $\Theta_1$ 은 차례로 1시간 전 $z$ · 1시간 전 잔차 · 24시간 전 $z$ · 24시간 전 잔차에 곱하는 수입니다.

## 이 노트북을 쓰는 법

| 무엇 | 어떻게 |
|---|---|
| 실행 환경 | CPU 로 충분합니다. ▶ 실행은 Google Colab 연결 안내를 따르고, AI 튜터는 상단 「Claude 도우미」를 설치한 뒤에 사용합니다 |
| 채점 준비 | Colab 이 연결되면 자동으로 준비됩니다. **코드 영역 위쪽에 「채점 준비됨」이 표시된 뒤에 미션을 시작하고, 미션 셀을 실행했는데 채점 결과가 출력되지 않으면 그 「채점 준비됨」을 누르세요** |
| 먼저 실행할 셀 | 맨 위 설치·폰트 셀(한 번이면 됩니다). 설치 뒤 numpy 관련 오류가 발생하면 런타임을 다시 시작하고 이 셀부터 다시 실행합니다 |
| 「준비 셀」 | 미션마다 첫 코드 셀입니다. 쓸 변수를 만드는 셀이라 고치지 않고 그대로 실행합니다 |
| 「PROMPT 셀」 | 미션마다 둘째 코드 셀입니다. `PROMPT` 를 내 말로 적어 AI 튜터에게 시키면 `# --- 여기부터 AI 코드 ---` 표시선 아래에 코드가 들어옵니다 |
| 출력 이름 | 미션 설명의 번호 목록에 굵게 적힌 이름을 글자 그대로 써서 「이름 = 값 단위」 한 행으로 출력합니다. 변수 이름은 자유이고, 채점은 이 출력 행과 그림을 봅니다 |
| 채점 | 실행하는 순간 실행 결과에 채점 결과가 출력됩니다. 채점 기준을 모두 충족하면 완성본 풀이가 열리고, 충족하지 못하면 어긋난 항목과 AI 튜터에 보낼 질문이 나옵니다 |
| 막혔을 때 | 같은 말을 두 번 적지 말고 `PROMPT` 에 빠진 조건을 하나 추가합니다. 그래도 안 되면 앞 대화를 이어서 쓰지 말고 요구를 한 덩어리로 다시 적어 새 메시지로 보냅니다 |

<details class="lms-extra"><summary>자세히 — 노트북을 내려받아 열었을 때</summary>

노트북을 내려받아 Colab 이나 내 컴퓨터에서 열었다면 맨 위의 접힌 `[채점]` 셀을 한 번 ▶ 실행한 뒤 시작합니다(코드는 접혀 있어도 실행됩니다).

</details>

In [ ]:
# 실습 준비 - 패키지 설치
!pip install -q "numpy>=1.26" "pandas>=2.0" "matplotlib>=3.7" "statsmodels==0.14.6"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.font_manager as fm
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
import warnings; warnings.filterwarnings("ignore")

# 한글 폰트 — Colab 이면 나눔고딕을 설치한 뒤 그 ttf 를 폰트 목록에 **직접** 등록한다.
#  설치는 디스크에 파일만 만든다. 이미 떠 있는 폰트 목록은 그 사실을 모르므로 addfont 로
#  넣어 줘야 이름이 보인다(넣지 않으면 DejaVu Sans 로 밀려 한글이 전부 □ 로 나온다).
import glob, shutil, subprocess
if shutil.which("apt-get"):
    subprocess.run(["apt-get", "-y", "-qq", "install", "fonts-nanum"], check=False)
for _p in glob.glob("/usr/share/fonts/truetype/nanum/Nanum*.ttf"):   # 설치 여부와 무관하게 한 번
    try: fm.fontManager.addfont(_p)
    except Exception: pass
try: fm._get_font.cache_clear()
except Exception: pass
installed = {f.name for f in fm.fontManager.ttflist}
FONT = next((n for n in ["Pretendard", "Apple SD Gothic Neo", "NanumGothic"] if n in installed), "DejaVu Sans")
plt.rcParams["font.family"] = FONT
plt.rcParams["axes.unicode_minus"] = FONT in {"Pretendard", "DejaVu Sans"}   # 나머지 폰트에는 U+2212 가 없다
%config InlineBackend.print_figure_kwargs = {"bbox_inches": None}   # 그림마다 폭이 깎이지 않게
plt.rcParams["figure.dpi"] = plt.rcParams["savefig.dpi"] = 150       # 폭 FIG_W x 150dpi = 1500px 고정
plt.rcParams["figure.autolayout"] = True
plt.rcParams.update({"font.size": 11, "axes.titlesize": 15, "axes.labelsize": 12,
                     "legend.fontsize": 11, "figure.titlesize": 17, "font.weight": "regular",
                     "text.color": "black", "axes.labelcolor": "black", "axes.titlecolor": "black",
                     "axes.edgecolor": "#757575", "xtick.color": "#757575", "ytick.color": "#757575",
                     "xtick.labelcolor": "black", "ytick.labelcolor": "black",
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False})

# 오늘 그림 전체가 함께 쓰는 색 — 기준선=남색, 관측=파랑, 강조=하늘색, 문맥=회색
COL = {"남색": "#051C2C", "파랑": "#2251FF", "하늘색": "#00A9F4",
       "회색": "#757575", "연회색": "#B3B3B3"}
FIG_W = 10.0     # 캔버스 폭은 오늘 그림 모두 이 값으로 고정하고 높이만 그림마다 조절한다


def datefmt(ax, fmt="%m-%d", n=5):
    """시간 축 눈금 개수와 표기를 잡아 날짜 라벨이 겹치지 않게 한다"""
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=n))
    ax.xaxis.set_major_formatter(mdates.DateFormatter(fmt))


def rmse(a, b):
    """두 계열의 평균제곱근오차 — ML 수업에서 쓰던 그 지표 그대로"""
    return float(np.sqrt(np.mean((np.asarray(a, dtype=float) - np.asarray(b, dtype=float)) ** 2)))


def mae(a, b):
    return float(np.mean(np.abs(np.asarray(a, dtype=float) - np.asarray(b, dtype=float))))

print("사용 폰트:", FONT)

In [ ]:
#@title [채점] 이 셀을 먼저 실행하세요 (한 번이면 됩니다)
#   미션의 「PROMPT 셀」을 실행하면 그 셀의 출력과 그림을 붙잡아 실행 결과에 채점 결과를 찍습니다.
#   통과하면 그 자리에서 완성본 풀이가 열리고, 아니면 항목별 힌트와 튜터에 보낼 질문이 나옵니다.
#   읽지 않고 실행만 해도 됩니다 — 기대값과 풀이는 이 셀에만 있고 「PROMPT 셀」에는 없습니다.
import base64
import glob
import io
import json
import re
import shutil
import subprocess
import sys

import matplotlib.font_manager as fm      # 아래 폰트 자급이 쓴다
import matplotlib.pyplot as plt


def _font_bootstrap():
    """한글 폰트 자급 — 맨 위 설치·폰트 셀을 건너뛰어도 그래프의 한글이 깨지지 않게.

    LMS 는 이 셀만 자동 실행하므로 폰트 준비도 여기서 한 번 더 한다. 설치·폰트 셀과 같은
    규칙이라 둘 다 돌아도 결과가 같고, apt-get 이 없는 자리(로컬)에서는 곧바로 지나간다.
    설치한 ttf 는 `addfont` 로 폰트 목록에 직접 넣는다 — 넣지 않으면 목록이 옛 상태라
    한글이 □ 로 깨진다(2026-09-09 Colab 실사고).
    """
    try:
        cands = ["Pretendard", "Apple SD Gothic Neo", "NanumGothic"]
        installed = {f.name for f in fm.fontManager.ttflist}
        if not any(n in installed for n in cands) and shutil.which("apt-get"):
            subprocess.run(["apt-get", "-y", "-qq", "install", "fonts-nanum"],
                           check=False, timeout=120)
        # 설치는 파일만 만든다 — 이미 떠 있는 폰트 목록에 addfont 로 넣어야 이름이 보인다.
        # (설치를 건너뛴 자리도 나눔 ttf 가 이미 있으면 이 줄들만으로 잡힌다.)
        for _p in glob.glob("/usr/share/fonts/truetype/nanum/Nanum*.ttf"):
            try: fm.fontManager.addfont(_p)
            except Exception: pass
        try: fm._get_font.cache_clear()
        except Exception: pass
        installed = {f.name for f in fm.fontManager.ttflist}
        _font = next((n for n in cands if n in installed), "DejaVu Sans")
        plt.rcParams["font.family"] = _font
        plt.rcParams["axes.unicode_minus"] = _font in {"Pretendard", "DejaVu Sans"}
    except Exception:
        print("한글 폰트를 못 찾아 그래프 글자가 깨질 수 있습니다 —"
              " 맨 위 설치·폰트 셀을 실행하세요")


_font_bootstrap()


def _dec_json(s):                          # base64(JSON) → 값. 리스트는 원래 튜플이라 되돌립니다
    def _t(v):
        return ({k: _t(x) for k, x in v.items()} if isinstance(v, dict)
                else tuple(_t(x) for x in v) if isinstance(v, list) else v)
    return _t(json.loads(base64.b64decode(s).decode("utf-8")))


# 채점 기준 — base64(JSON), 통과 판단에만 쓴다
_ANS = _dec_json(
           "eyJtMSI6IHsidmFsdWVzIjogeyLqsoDspp0gMTY47Iuc6rCEIFJNU0UiOiBbMzg1LjYsIDM4Ni42LCAi64yAIl0sICIx7KO8"
           "7J28IOyghCDqsJIg6rec7LmZ7J2YIFJNU0UiOiBbMjA1LjE1LCAyMDUuMzUsICLrjIAiXX0sICJwbG90IjogeyJraW5kIjog"
           "ImxpbmUiLCAibl9zZXJpZXMiOiBbMiwgNF0sICJuX3BvaW50cyI6IFsxNjgsIDIwMDBdLCAi652gIjogdHJ1ZSwgIuy2leud"
           "vOuyqCI6IHRydWV9fSwgIuqzvOygnCI6IHsidmFsdWVzIjogeyLsnbjsiJjsnbjqs4TtkZwg7ZaJIOyImCI6IFsxNjcuNSwg"
           "MTY4LjUsICLtlokiXSwgIu2ZleyLoOycvOuhnCDrgqjsnYAg7Iuc6rCBIOyImCI6IFswLjUsIDE2Ny41LCAi7Iuc6rCEIiwg"
           "Iu2ZleyLoCJdLCAi67O066WY66GcIOuCqOydgCDsi5zqsIEg7IiYIjogWzAuNSwgMTY3LjUsICLsi5zqsIQiLCAi67O066WY"
           "Il0sICLtlZjtlZzsnbQg7J2M7IiY7J24IOyLnOqwgSDsiJgiOiBbNTQuNSwgNTUuNSwgIuyLnOqwhCJdLCAi7J6U7LCoIExq"
           "dW5nLUJveCDsi5zssKggMjQg7Ya16rOE65+JIjogWzI3NS4wLCAyODAuMF0sICLsnpTssKggTGp1bmctQm94IOyLnOywqCAy"
           "NCBwIOqwkiI6IFsxZS02MCwgMWUtMzBdfSwgInBsb3QiOiB7ImtpbmQiOiAibGluZSIsICJuX3NlcmllcyI6IFsxLCA0XSwg"
           "Im5fcG9pbnRzIjogWzE2OCwgMjAwMF0sICLrnaAiOiB0cnVlLCAidmxpbmVfeCI6IFswLjAsIDJlKzE4XSwgIuy2leudvOuy"
           "qCI6IHRydWV9fX0=")
_REVEAL = {                                # 완성본 풀이 — base64, 통과할 때만 디코드해 인쇄합니다
    "m1": ("8J+UkyDrr7jshZggMSDsmYTshLHrs7gg4oCUIEFJIOy9lOuTnCDtkZzsi5zshKAg7JWE656Y7JeQIOuTpOyWtOyZgOyVvCDtlaAg7L2U65Oc64qUIOydt"
           "CDsoJXrj4TsnoXri4jri6QKCuKWoCDsnbQg7L2U65Oc66W8IOu2gOuluCDtlITroaztlITtirgg7JiICgogICAg7J2066+4IOunjOuTpOyWtOyguCDsno"
           "jripQgZGYsIHlfdHJhaW4sIHlfdmFsaWQsIE9SREVSLCBTRUFTT05BTF9PUkRFUiwgU0FSSU1BWCwgcm1zZSwgZGF0ZWZtdCDrpbwg6re464yA66GcIOy"
           "TsOqzoCDri6Tsi5wg66eM65Ok7KeA64qUCiAgICDrp5DslYQg7KSYIOKAlCBybXNlKOyLpOy4oSwg7JiI7LihKSDripQg7KCc6rOx6re8IO2Pieq3oOyg"
           "nOqzseyYpOywqOulvCDrsJjtmZjtlZjqs6AgZGF0ZWZtdCjstpUsIO2RnOq4sCwg64iI6riIIOyImCkg64qUIOqwgOuhnOy2lSDrgqDsp5wg7ZGc6riw6"
           "6W8IOygleumrO2VmOuKlCDtlajsiJjslbwuIO2VmeyKtSDqtazqsIQKICAgIHlfdHJhaW4g7JeQIE9SREVSIOyZgCBTRUFTT05BTF9PUkRFUiDrpbwg6r"
           "e464yA66GcIOyghOuLrO2VtCBTQVJJTUFYIOulvCDtlZwg67KIIOygge2Vqe2VtCDspJgg4oCUIOyYteyFmOydgCB0cmVuZCDsl4bsnYzsl5Ag7KCV7IO"
           "B7ISxwrfqsIDsl63shLEg6rCV7KCc7JW8LiDsoIHtlantlZwKICAgIOuqqO2YleyXkOyEnCDqsoDspp0g6ri47J2066eM7YG87J2YIOyYiOy4oeydhCDt"
           "lZwg67KI7JeQIOuwm+yVhCA5NSUg7JiI7Lih6rWs6rCE6rmM7KeAIOy2lOy2nO2VmOqzoCwg6rKA7KadIOyLnOqwgeydhCDsnbjrjbHsiqTroZwg7Iuk7"
           "LihIMK3IOyYiOy4oSDCtyDqtazqsIQg7ZWY7ZWcIMK3IOq1rOqwhCDsg4HtlZwg64SkIOyXtOydhCDri7TsnYAg7ZGc66W8CiAgICDrp4zrk6TslrQg7K"
           "SYIOKAlCDqtazqsIQg65GQIOyXtOydgCDsnbTrpoTsnbQg7JWE64uI6528IOychOy5mOuhnCDstpTstpztlZjrqbQg7JWI7KCE7ZW0LiDqt7jrpqzqs6A"
           "gZGYg7J2YIOuztOyglSDsiJjsmpQgZGVtYW5kIOulvCAxNjjsi5zqsIQg65Kk66GcIOydtOuPmeyLnOy8nCDqsoDspp0g7Iuc6rCB66eMIOy2lOy2nO2V"
           "nCDqs4TsoIgKICAgIOuCmOydtOu4jCAxNjgg64+EIOunjOuTpOyWtCDspJguICLqsoDspp0gMTY47Iuc6rCEIFJNU0UgPSAo7Iir7J6QKSDrjIAiIOyZg"
           "CAiMeyjvOydvCDsoIQg6rCSIOq3nOy5meydmCBSTVNFID0gKOyIq+yekCkg64yAIiDrpbwg7IaM7IiY7KCQIOyVhOuemCAy7J6Q66as66GcIO2VnCDtlo"
           "nslKkgcHJpbnQg7ZW0CiAgICDspJguIOq3uOuemO2UhOuKlCDrlLEg7ZWcIOyepeunjCDigJQg6rKA7KadIOq1rOqwhOydhCDqsIDroZzstpXsl5Ag65G"
           "Q6rOgIOq1rOqwhCDtlZjtlZzqs7wg7IOB7ZWcIOyCrOydtOulvCDsmIXsnYAg7IOJ7Jy866GcIOyxhOyatCDrkqQg7Iuk7LihIOyEoOqzvCDsmIjsuKEg"
           "7ISg7J2EIOqyueyzkCDqt7jrpqzqs6AsIOuRkCDshKDsl5Ag6rCB6rCBCiAgICAi7Iuk7LihIsK3IuyYiOy4oSIg7J2066aE7J2EIOu2gOyXrO2VtCDrs"
           "pTroYDrpbwg7ZGc7Iuc7ZWY6rOgLCDqsIDroZzstpXqs7wg7IS466Gc7LaV7JeQIOudvOuyqOydhCDrtoDsl6ztlZjqs6Ag6rCA66Gc7LaVIOuCoOynnC"
           "DtkZzquLDrpbwg7KCV66as7ZW0IOykmC4KCiAgICByZXMgPSBTQVJJTUFYKHlfdHJhaW4sIG9yZGVyPU9SREVSLCBzZWFzb25hbF9vcmRlcj1TRUFTT05"
           "BTF9PUkRFUiwKICAgICAgICAgICAgICAgICAgdHJlbmQ9Tm9uZSwgZW5mb3JjZV9zdGF0aW9uYXJpdHk9VHJ1ZSwKICAgICAgICAgICAgICAgICAgZW5m"
           "b3JjZV9pbnZlcnRpYmlsaXR5PVRydWUpLmZpdChkaXNwPUZhbHNlKSAgICAgIyDspIDruYQg7IWA7J20IOy2nOugpe2VnCDshLgg7Ji17IWYIOq3uOuMg"
           "OuhnAoKICAgIGYgPSByZXMuZ2V0X2ZvcmVjYXN0KHN0ZXBzPWxlbih5X3ZhbGlkKSkgICAgICAgIyDtlZwg67KI66eMIO2YuOy2nO2VnOuLpCDigJQgMT"
           "Y47Iuc6rCE7J2EIO2VnCDrsojsl5Ag7IKw7Lac7ZWc64ukCiAgICBjaSA9IGYuY29uZl9pbnQoYWxwaGE9MC4wNSkgICAgICAgICAgICAgICAgICAgICM"
           "gMC4wNSA9IDk1JSDsmIjsuKHqtazqsIQKICAgIGZjID0gcGQuRGF0YUZyYW1lKHsiYWN0dWFsIjogeV92YWxpZC52YWx1ZXMsICJwcmVkIjogZi5wcmVk"
           "aWN0ZWRfbWVhbi52YWx1ZXMsCiAgICAgICAgICAgICAgICAgICAgICAgImxvOTUiOiBjaS5pbG9jWzosIDBdLnZhbHVlcywgImhpOTUiOiBjaS5pbG9jW"
           "zosIDFdLnZhbHVlc30sCiAgICAgICAgICAgICAgICAgICAgICBpbmRleD15X3ZhbGlkLmluZGV4KSAgICAgICAgICMg6rWs6rCEIOuRkCDsu6zrn7zsnY"
           "Ag7J2066aE7J20IOyVhOuLiOudvCDsnITsuZjroZwg7LaU7Lac7ZWc64ukCgogICAgc24xNjggPSBkZlsiZGVtYW5kIl0uc2hpZnQoMTY4KS5sb2NbeV9"
           "2YWxpZC5pbmRleF0gICAgICAjIOqzhOygiCDrgpjsnbTruIwgMTY4IOKAlCAx7KO87J28IOyghCDqsJnsnYAg7Iuc6rCBIOqwkgoKICAgIHByaW50KGYi"
           "6rKA7KadIDE2OOyLnOqwhCBSTVNFID0ge3Jtc2UoZmNbJ2FjdHVhbCddLCBmY1sncHJlZCddKTouMmZ9IOuMgCIpCiAgICBwcmludChmIjHso7zsnbwg7"
           "KCEIOqwkiDqt5zsuZnsnZggUk1TRSA9IHtybXNlKHlfdmFsaWQsIHNuMTY4KTouMmZ9IOuMgCIpCgogICAgZmlnLCBheCA9IHBsdC5zdWJwbG90cyhmaW"
           "dzaXplPShGSUdfVywgMy44KSkKICAgIGF4LmZpbGxfYmV0d2VlbihmYy5pbmRleCwgZmNbImxvOTUiXSwgZmNbImhpOTUiXSwgY29sb3I9Q09MWyLtlZj"
           "ripjsg4kiXSwgYWxwaGE9MC4xOCkKICAgIGF4LnBsb3QoZmMuaW5kZXgsIGZjWyJhY3R1YWwiXSwgY29sb3I9Q09MWyLrgqjsg4kiXSwgbHc9MS40LCBs"
           "YWJlbD0i7Iuk7LihIikKICAgIGF4LnBsb3QoZmMuaW5kZXgsIGZjWyJwcmVkIl0sIGNvbG9yPUNPTFsi7ZWY64qY7IOJIl0sIGx3PTEuOCwgbGFiZWw9I"
           "uyYiOy4oSIpCiAgICBheC5sZWdlbmQobG9jPSJ1cHBlciByaWdodCIsIGZyYW1lb249RmFsc2UpCiAgICBheC5zZXRfeGxhYmVsKCLqsoDspp0g6rWs6r"
           "CEIOyLnOqwgSIpCiAgICBheC5zZXRfeWxhYmVsKCLsiJjsmpQo64yAKSIpCiAgICBkYXRlZm10KGF4LCAiJW0tJWQiLCA4KQogICAgcGx0LnNob3coKQo"
           "K7ZW07ISdCgotIOyYiOyDgSAz6rCcIOykkSDri7XsnYAg65GQIOuyiOynuCwg6rOE7KCIIOuCmOydtOu4jCAxNjgg7J6F64uI64ukLiDqsoDspp0gMTY4"
           "7Iuc6rCE7JeQ7IScIFNBUklNQSDrqqjtmJXsnYAgUk1TRSAzODYuMDjrjIAsIOqzhOygiCDrgpjsnbTruIwgMTY4IOydgCAyMDUuMjXrjIDsnoXri4jri"
           "6QuIDEsODQ47Iuc6rCE7Jy866GcIOqzhOyImCA06rCc66W8IOy2lOygle2VnCDrqqjtmJXsnbjrjbAsIOqzhOyImOulvCDtlZjrgpjrj4Qg7LaU7KCV7Z"
           "WY7KeAIOyViuuKlCDqt5zsuZnsnZgg7Jik7LCo6rCAIOq3uCDsoIjrsJjsnbTsl4jsirXri4jri6QuIO2VmOujqCDri6jsnITroZwg64KY64igIOuztOy"
           "VhOuPhCA37J28IOyghOu2gOyXkOyEnCDqt5zsuZkg7Kq9IOyYpOywqOqwgCDrjZQg7J6R7Iq164uI64ukLiDslrTsoJwg7LCo7IiY7J2YIFJNU0Ug64qU"
           "IDI3MS4yMuuMgOyYgOyKteuLiOuLpC4gQUlDIOuKlCA3MDYuNjUg64K066Ck6rCU64qU642wIOqygOymnSAxNjjsi5zqsIQg7Jik7LCo64qUIDExNeuMg"
           "CDsu6TsoYzsirXri4jri6QuCi0g7Jik7LCo6rCAIOuCnCDqs7PsnYAg7IiY7KSA7J6F64uI64ukLiDsmIjsuKHshKDsnYAgOOyLnOyZgCAxOOyLnOyXkC"
           "Drp47snYAg7ZWY66OoIOuqqOyWkeydhCDrlLDrnbzqsJTsp4Drp4wsIO2VmOujqCDtj4nqt6DsnLzroZwg67O066m0IDfsnbwg66qo65GQIOyYiOy4oey"
           "dtCDsi6TsuKHrs7Tri6Qg64aS7JWY7Iq164uI64ukLiDsmIjsuKEg7Y+J6reg7J20IDk2NS4264yA7J24642wIOyLpOy4oSDtj4nqt6DsnYAgNjk0LjLr"
           "jIDroZwg7LCo7J206rCAICsyNzEuNOuMgOyeheuLiOuLpC4g7J20IOuqqO2YleydmCDqs4TsoIgg7J2064+Z7Y+J6regIOqzhOyImCDOmOKCgSDsnYAg4"
           "oiSMS4wIOyXkCDri7/slYQsIOuovCDsp4Dtj4nsnZgg7JiI7Lih7J20IO2VmeyKtSAxMeyjvOydmCDsi5zqsIHrs4Qg7Y+J6regIOuqqOyWkeycvOuhnC"
           "Drj4zslYTqsJHri4jri6QuIOuniOyngOuniSDrgqAg7JiI7Lih6rO8IOq3uCDtj4nqt6DsnZgg7LCo7J2064qUIOyLnOqwgeuniOuLpCAxOOuMgCDslYj"
           "sqr3snoXri4jri6QuIOqygOymnSDso7zqsITsnZgg7Iuk7KCcIOyImOykgOydgCDqt7gg7Y+J6reg67O064ukIOyVhOuemOyYgOyKteuLiOuLpC4KLSDq"
           "t7gg7ZWY65297J2AIOq3nOy5mSDsqr0g7JiI7Lih7JeQIOyekOuPmeycvOuhnCDrsJjsmIHrkKnri4jri6QuIDHso7zsnbwg7KCEIOyLpOy4oeydmCDtj"
           "4nqt6DsnYAgODMwLjLrjIDroZwg7J2066+4IOuCtOugpOyYqCDqsJLsnoXri4jri6QuIOyImOykgOydhCDrgq7stpgg7JqU7J247J2AIOq4sOyYqOydtO"
           "qzoCwg6rKA7KadIOyjvOqwhCDtj4nqt6AgNC45MeuPhOuhnCAx7KO87J28IOyghCA5LjM364+E67O064ukIDQuNeuPhCDrgq7slZjsirXri4jri6QuCi0"
           "g7JiI7Lih6rWs6rCE7J2AIOq4sOuMgO2VnCDsiJjspIDsnYQg7Lap7KGx7ZaI7Iq164uI64ukLiA5NSUg7JiI7Lih6rWs6rCE7J20IDE2OOyLnOqwhCDs"
           "pJEgMTYy7Iuc6rCE7JeQ7IScIOyLpOy4oeydhCDri7TslYQg66qF66qpIOyImOykgOyXkCDqsIDquZ3qs6AsIOq1rOqwhCDquLjsnbTripQg7LKrIOyLn"
           "OqwgSA3NjYuOOuMgOyXkOyEnCDssqvrgqAg64GdIDEsNTUwLjPrjIDroZwg6ri47Ja07KeR64uI64ukLiDqt7gg65Kk66Gc64qUIOuRmOynuCDrgqAg64"
           "GdIDEsNjIwLjXrjIAsIOuniOyngOuniSAxLDYyOS4x64yA66GcIOyhsOq4iOyUqeunjCDriqHri4jri6QuIOuLpOunjCDtlZjtlZzsnbQg7J2M7IiY7J2"
           "4IOyLnOqwgeydtCA1NeqwnOyeheuLiOuLpC4g7IiY7JqU64qUIOydjOyImOqwgCDrkKAg7IiYIOyXhuycvOuLiCDqt7gg7Iuc6rCB7J2EIOyWtOuWu+qy"
           "jCDrs7Tqs6DtlaDsp4DqsIAg7Jik64qYIOqzvOygnOydmCDtjJDri6gg7ZWY64KY7J6F64uI64ukLgotIOyYpOuKmCDtmZXsnbjtlZwg6rKD7J2AIOOAj"
           "OydtCDsp4Dtj4ksIOydtCDqtazqsIQsIOydtCDsobDqsbTsl5DshJwg6rec7LmZ7J2YIOyYpOywqOqwgCDrjZQg7J6R7JWY64uk44CNIOydtOyngCDjgI"
           "zrqqjtmJXsnbQg6rec7LmZ67O064ukIOuqu+2VmOuLpOOAjSDqsIAg7JWE64uZ64uI64ukLiDqt5zsuZnsnYAgMeyjvOydvCDsoITqs7wg64uk66W4IOy"
           "dvOydtCDrsJzsg53tlZwg64Kg7J2EIOunnu2eiOyngCDrqrvtlanri4jri6QuIFNBUklNQSDrqqjtmJXsl5Drj4Qg6re4IOygleuztOqwgCDsl4bsl4js"
           "p4Drp4wsIOyeheugpe2VoCDqsr3roZzrpbwg66eM65Oc64qUIOqyg+ydtCDri6TshK/sp7gg64Kg7J2YIOydvOyeheuLiOuLpC4KCuy9lOuTnOqwgCDri"
           "7XsnYQg66eM65Oc64qUIOuwqeyLnSDigJQg7KCE7J6EIOy9lOuTnOyZgCDqs6DsuZwg7L2U65Oc7J2YIOywqOydtOuKlCBzaGlmdCDsl5Ag7KSAIOyLnO"
           "qwhCDsiJgg7ZWY64KY7J6F64uI64ukLiDsoITsnoQg7L2U65Oc7J2YIDI0NS40NeuMgOuKlCDthqDsmpTsnbzsnbggMTHsm5QgMTfsnbwgMOyLnOyXkCD"
           "quIjsmpTsnbwgMTbsnbwgMOyLnCDqsJLsnYQg7JO0IOqysOqzvOyeheuLiOuLpC4g6rOg7LmcIGBzaGlmdCgxNjgpLmxvY1tpbmRleF1gIOuKlCAxMeyb"
           "lCAxN+ydvCAw7Iuc7JeQIO2VnCDso7wg7KCEIO2GoOyalOydvOyduCAxMeyblCAxMOydvCAw7Iuc7J2YIOyLpOy4oeydhCDslIHri4jri6QuIOq3uCDqs"
           "JLsnYAg7KCE67aAIO2VmeyKtSDqtazqsIQg7JWI7J206528IOuvuOuemOulvCDrs7Tsp4Ag7JWK7Iq164uI64ukLiBTQVJJTUEg7Kq9IGBnZXRfZm9yZW"
           "Nhc3Qoc3RlcHM9MTY4KWAg64+EIO2VmeyKtSDqtazqsIQg64Gd7J2YIOyDge2DnCDtlZjrgpjroZwgMTY47Iuc6rCE7J2EIO2VnCDrsojsl5Ag6rWs7ZW"
           "p64uI64ukLiDrkZAg7JiI7Lih7J20IOqwmeydgCAxNjjsi5zqsIQg7JWeIOyngO2PiSwg6rCZ7J2AIOqygOymnSDqtazqsITsl5DshJwg7Lih7KCV65CY"
           "7JeI6riwIOuVjOusuOyXkCDsnbQg67mE6rWQ6rCAIOyEseumve2VqeuLiOuLpC4="),
    "과제": ("8J+UkyDqs7zsoJwg7JmE7ISx67O4IOKAlCBBSSDsvZTrk5wg7ZGc7Iuc7ISgIOyVhOuemOyXkCDrk6TslrTqsIDslbwg7ZWgIOy9lOuTnOuKlC"
           "DsnbQg7KCV64+E7J6F64uI64ukICjqtazrtoQg6riw7KSA6rCS7J2AIOyYiOyLnOyeheuLiOuLpCkKCuKWoCDsnbQg7L2U65Oc66W8IOu2gOul"
           "uCDtlITroaztlITtirgg7JiICgogICAg7J2066+4IOunjOuTpOyWtOyguCDsnojripQgZmMsIHJlc2lkLCB5X3ZhbGlkLCBkZiwgYWNvcnJfbG"
           "p1bmdib3gsIHJtc2UsIG1hZSwgZGF0ZWZtdCDrpbwg6re464yA66GcIOyTsOqzoCDri6Tsi5wg7KCB7ZWp7ZWY7KeA64qUIOunkOyVhCDspJgg"
           "4oCUCiAgICBmYyDripQg6rKA7KadIDE2OOyLnOqwhOydhCDsnbjrjbHsiqTroZwg7Iuk7LihIGFjdHVhbCDCtyDsmIjsuKEgcHJlZCDCtyDqta"
           "zqsIQg7ZWY7ZWcIGxvOTUgwrcg6rWs6rCEIOyDge2VnCBoaTk1IMK3IOq1rOqwhCDquLjsnbQgd2lkdGgg66W8IOuLtOydgCDtkZzsnbTqs6Ag"
           "cmVzaWQg64qUCiAgICDqt7gg66qo7ZiV7J2YIOyelOywqOyVvC4g64KY64qUIOq1rOu2hCDquLDspIDqsJLsnYQgMTYwMOuMgOuhnCDsoJXtlo"
           "jslrQg4oCUIGZjIOulvCDrs7XsgqztlbQg6rWs6rCEIOq4uOydtOqwgCAxNjAw64yA67O064ukIOynp+ydgCDsi5zqsIHsnYAg44CM7ZmV7Iug"
           "44CNLCDqt7jroIfsp4AKICAgIOyViuydgCDsi5zqsIHsnYAg44CM67O066WY44CNIOuhnCDrk7HquIkg7Je07J2EIOy2lOqwgO2VtCDspJguIO"
           "q3uCDri6TsnYwgIuyduOyImOyduOqzhO2RnCDtlokg7IiYID0gKOyIq+yekCkg7ZaJIiDtlZwg7ZaJLCAi7ZmV7IugID0gKOyIq+yekCkg7Iuc"
           "6rCEIMK3IOuztOulmCA9ICjsiKvsnpApIOyLnOqwhCIg7J2EIO2VnAogICAg7ZaJLCAi7ZWY7ZWc7J20IOydjOyImOyduCDsi5zqsIEg7IiYID"
           "0gKOyIq+yekCkg7Iuc6rCEIiDsnYQg7ZWcIO2WiSBwcmludCDtlbQg7KSYLiDsnbTslrTshJwgcmVzaWQg7JeQIGFjb3JyX2xqdW5nYm94IOul"
           "vCDsi5zssKggMjQgwrcgbW9kZWxfZGYgNCDroZwg7KCB7Jqp7ZW0CiAgICAi7J6U7LCoIExqdW5nLUJveCDsi5zssKggMjQg7Ya16rOE65+JID"
           "0gKOyIq+yekCkiIOulvCDshozsiJjsoJAg7JWE656YIDLsnpDrpqzroZwsICLsnpTssKggTGp1bmctQm94IOyLnOywqCAyNCBwIOqwkiA9ICjs"
           "iKvsnpApIiDrpbwg7Jyg7Zqo7Iir7J6QIOyEuCDsnpDrpqzroZwg7ZWcIO2WieyUqQogICAg7Lac66Cl7ZW0IOyjvOqzoC4g6re4656Y7ZSE64"
           "qUIOuUsSDtlZwg7J6l66eMIOKAlCDqsoDspp0g6rWs6rCEIOyLnOqwgeydhCDqsIDroZzstpXsl5Ag65GQ6rOgIGxvOTUg7JmAIGhpOTUg7IKs"
           "7J2066W8IOyYheydgCDsg4nsnLzroZwg7LGE7Jq0IOuSpCDsmIjsuKEg7ISg6rO8IOyLpOy4oSDshKDsnYQg6rK57LOQIOq3uOumrOqzoCwKIC"
           "AgIO2ZleyLoOydtCDrgZ3rgpjqs6Ag7LKY7J2MIOuztOulmOqwgCDrkJjripQg7Iuc6rCB7JeQIOyEuOuhnCDsoJDshKDsnYQg7ZWY64KYIOq4"
           "i+qzoCDqsIDroZzstpXqs7wg7IS466Gc7LaV7JeQIOudvOuyqOydhCDrtoDsl6ztlZjqs6Ag6rCA66Gc7LaVIOuCoOynnCDtkZzquLDripQgZG"
           "F0ZWZtdCjqt7gg7LaVLCAiJW0tJWQiLCA4KSDsspjrn7wg7LaV6rO8CiAgICDtkZzquLDsmYAg64iI6riIIOyImOulvCDsoITri6ztlbQg7KCV"
           "66as7ZW0IOykmC4KCiAgICB3aWR0aF9saW1pdCA9IDE2MDAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIOuCtOqwgCDqs6"
           "Drpbgg6rWs67aEIOq4sOykgOqwkiDigJQg6rWs6rCEIOq4uOydtOulvCDrjIAg64uo7JyE66GcIOuzuOuLpAogICAgaGFuZG92ZXIgPSBmYy5j"
           "b3B5KCkKICAgIGhhbmRvdmVyWyLrk7HquIkiXSA9IG5wLndoZXJlKGhhbmRvdmVyWyJ3aWR0aCJdIDwgd2lkdGhfbGltaXQsICLtmZXsi6AiLC"
           "Ai67O066WYIikKCiAgICBwcmludChmIuyduOyImOyduOqzhO2RnCDtlokg7IiYID0ge2xlbihoYW5kb3Zlcil9IO2WiSIpCiAgICBwcmludChm"
           "Iu2ZleyLoCA9IHtpbnQoKGhhbmRvdmVyWyfrk7HquIknXSA9PSAn7ZmV7IugJykuc3VtKCkpfSDsi5zqsIQiCiAgICAgICAgICBmIiAgIMK3IC"
           "Ag67O066WYID0ge2ludCgoaGFuZG92ZXJbJ+uTseq4iSddID09ICfrs7TrpZgnKS5zdW0oKSl9IOyLnOqwhCIpCiAgICBwcmludChmIu2VmO2V"
           "nOydtCDsnYzsiJjsnbgg7Iuc6rCBIOyImCA9IHtpbnQoKGhhbmRvdmVyWydsbzk1J10gPCAwKS5zdW0oKSl9IOyLnOqwhCIpCgogICAgbGIgPS"
           "BhY29ycl9sanVuZ2JveChyZXNpZCwgbGFncz1bMjRdLCBtb2RlbF9kZj00KQogICAgcHJpbnQoZiLsnpTssKggTGp1bmctQm94IOyLnOywqCAy"
           "NCDthrXqs4Trn4kgPSB7ZmxvYXQobGJbJ2xiX3N0YXQnXS5pbG9jWzBdKTouMmZ9IikKICAgIHByaW50KGYi7J6U7LCoIExqdW5nLUJveCDsi5"
           "zssKggMjQgcCDqsJIgPSB7ZmxvYXQobGJbJ2xiX3B2YWx1ZSddLmlsb2NbMF0pOi4zZ30iKQoKICAgIGJvdW5kYXJ5ID0gaGFuZG92ZXIuaW5k"
           "ZXhbaGFuZG92ZXJbIuuTseq4iSJdLmVxKCLrs7TrpZgiKV1bMF0gICAgICAgICAgICMg7ZmV7Iug7J20IOuBneuCmOuKlCDssqsg7Iuc6rCBCi"
           "AgICBmaWcsIGF4ID0gcGx0LnN1YnBsb3RzKGZpZ3NpemU9KEZJR19XLCAzLjgpKQogICAgYXguZmlsbF9iZXR3ZWVuKGhhbmRvdmVyLmluZGV4"
           "LCBoYW5kb3ZlclsibG85NSJdLCBoYW5kb3ZlclsiaGk5NSJdLCBjb2xvcj1DT0xbIu2VmOuKmOyDiSJdLCBhbHBoYT0wLjE4KQogICAgYXgucG"
           "xvdChoYW5kb3Zlci5pbmRleCwgaGFuZG92ZXJbInByZWQiXSwgY29sb3I9Q09MWyLtlZjripjsg4kiXSwgbHc9MS44KQogICAgYXgucGxvdCho"
           "YW5kb3Zlci5pbmRleCwgaGFuZG92ZXJbImFjdHVhbCJdLCBjb2xvcj1DT0xbIuuCqOyDiSJdLCBsdz0xLjApCiAgICBheC5heHZsaW5lKGJvdW"
           "5kYXJ5LCBjb2xvcj1DT0xbIuuCqOyDiSJdLCBscz0iLS0iLCBsdz0xLjIpCiAgICBheC5zZXRfeGxhYmVsKCLqsoDspp0g6rWs6rCEIOyLnOqw"
           "gSIpCiAgICBheC5zZXRfeWxhYmVsKCLsiJjsmpQo64yAKSIpCiAgICBkYXRlZm10KGF4LCAiJW0tJWQiLCA4KQogICAgcGx0LnNob3coKQoK7Z"
           "W07ISdCgotIOq1rOqwhCDquLjsnbTsnZgg6rWs67aEIOq4sOykgOqwkuydhCAxLDYwMOuMgOuhnCDsoJXtlZjrqbQg7ZmV7Iug7J20IDI27Iuc"
           "6rCELCDrs7TrpZjqsIAgMTQy7Iuc6rCE7J6F64uI64ukLiDrkZAg7IiY66W8IOuNlO2VmOuptCDtkZzsnZggMTY47ZaJ6rO8IOqwmeyKteuLiO"
           "uLpC4g6rWs6rCEIOq4uOydtOuKlCDsp4Dtj4nsnbQg66mA7Ja07KeI7IiY66GdCiAg7Luk7KeA66+A66GcIOqyveqzhOqwgCDsi5zqsIQg7LaV"
           "7J2EIO2VnCDrsojrp4wg64KY64iE6rOgLCDqt7jrprzsl5DripQg7IS466Gc7ISgIO2VmOuCmOqwgCDrgqjsirXri4jri6QuCi0g44CM7J6Q7I"
           "S47Z6IIOKAlCDqtazqsIQg6ri47J207JmAIOyDgeuMgCDqtazqsIQg6ri47J206rCAIOunjOuTnOuKlCDshJzroZwg64uk66W4IOqyveqzhOOA"
           "jeyXkOyEnCDrjIDsobDtlZwg64yA66GcLCDqsJnsnYAg7ZGc66W8IOyDgeuMgCDqtazqsIQg6ri47J20IDEuNSDroZwg6rWs67aE7ZWY66m0IO"
           "2ZleyLoOydtCA3MeyLnOqwhOydtCDrkKnri4jri6QuIOydtOuyiCDqsr3qs4TripQg7Iuc6rCB7J2EIOuUsOudvCDsg53quYHri4jri6Qg4oCU"
           "IOyYiOy4oeqwkuydtCDsnpHsnYAg7IOI67K97J20IOuztOulmOuhnAogIOu2hOulmOuQmOqzoCDrgq4g7Iuc6rCE64yA6rCAIO2ZleyLoOycvO"
           "uhnCDrgqjsirXri4jri6QuIOq1rOu2hO2VmOuKlCDsp4DtkZwg7ZWY64KY66W8IOuwlOq/qOydhCDrv5DsnbjrjbAg67Cw7LCo7YyA7J20IOuw"
           "m+uKlCDtkZzqsIAg64us65287KeR64uI64ukLgotIO2VmO2VnOydtCDsnYzsiJjsnbgg7Iuc6rCB7J2AIDU16rCc7J6F64uI64ukLiDqtazqsI"
           "TsnYQg7KCV6rec67aE7Y+s66GcIOqzhOyCsO2VmOuvgOuhnCDsmIjsuKHqsJLsnbQg7J6R6rOgIOq1rOqwhOydtCDquLQg7IOI67K97JeQIOyV"
           "hOuemOyqveydtCAwIOuwkeycvOuhnCDrgrTroKTqsJHri4jri6QuIDAg7Jy866GcCiAg64yA7LK07ZWY66m0IO2RnOuKlCDsoJXrpqzrkJjsp4"
           "Drp4wg66qo7ZiV7J20IOqzhOyCsO2VnCDrtojtmZXsi6TshLHsnYQg6rCQ7LaU6rOgLCDruYTsm4wg65GQ66m0IOuwsOywqO2MgOydtCDsgqzs"
           "mqntlaAg7Iuc6rCB7J20IOykhOyWtOuTreuLiOuLpC4g7Ja064qQIOyqveydtOuToCDtmozsi6Dsl5Ag7ZWcIOusuOyepeycvOuhnCDrsJ3tnp"
           "nri4jri6QuCi0g7J6U7LCoIExqdW5nLUJveCDripQg7Iuc7LCoIDI0IOyXkOyEnCDthrXqs4Trn4kgMjc3LjQ2LCBwIOqwkiAzLjE2ZS00NyDr"
           "oZwg6reA66y06rCA7ISk7J2EIOq4sOqwge2VqeuLiOuLpC4g64Ko7J2AIOyekOq4sOyDgeq0gOydtCDsnojri6TripQg65y77J206rOgLCDqt7"
           "jrn6zrqbQg7J20IOq1rOqwhOydgAogIOOAjOuCqOydgCDqsoPsnbQg67Cx7IOJ7J6h7J2M67+Q44CNIOydtOudvOuKlCDsoITsoJzrpbwg65GQ"
           "6rOgIOqzhOyCsO2VnCDquLjsnbTrnbwg7J2066Gg7KCB7Jy866Gc64qUIOyLpOygnOuztOuLpCDsp6fsnYQg7IiYIOyeiOyKteuLiOuLpC4g64"
           "uk66eMIOydtOuyiCAxNjjsi5zqsITsl5DshJzripQg7JiI7Lih6rWs6rCE7J20CiAgMTYy7Iuc6rCE7JeQ7IScIOyLpOy4oeydhCDri7TslZjs"
           "nLzri4gsIO2RnOulvCDtj5DquLDtlaAg7J207Jyg6rCAIOyVhOuLiOudvCDtlZzqs4TroZwg7KCB7Ja0IOyghOuLrO2VoCDrgrTsmqnsnoXri4"
           "jri6QuCi0g6re4656Y7IScIO2ajOyLoOydgCDsnbTroIfqsowg65Cp64uI64ukLiDsnbTrsogg7KO8IOuwsOywqCDrjIDsiJjripQg6rOE7KCI"
           "IOuCmOydtOu4jCAxNjgg7JeQ7IScIOqwgOyguOyYteuLiOuLpC4g7JWeIDI27Iuc6rCE7J2AIOq3uCDrjIDsiJjrpbwg6riI7JqU7J287JeQCi"
           "Ag7ZmV7KCV7ZWY64qUIO2ZleyLoOycvOuhnCwg64KY66i47KeA64qUIOuLueydvCDslYTsuagg7Iuk7Lih7J2EIOuztOqzoCDri6Tsi5wg7KCV"
           "7ZWY64qUIOuztOulmOuhnCDsoIHsirXri4jri6QuIO2VmO2VnOydtCDsnYzsiJjsnbgg7Iuc6rCB7J2AIDAg7Jy866GcCiAg7KCB6rOgLCDsm5"
           "Drnpgg6rOE7IKw65CcIO2VmO2VnCDqsJLsnYQg7JiG7JeQIO2VqOq7mCDsoIHslrQgMCDsnLzroZwg7J6Y656Q64uk64qUIOyCrOyLpOydhCDr"
           "gqjquYHri4jri6QuIOq1rOu2hCDquLDspIDqsJLqs7wg6re4IOqwkuydhCDsoJXtlZwg7Jq07JiBIOq3vOqxsOulvCDtlajqu5gg67Cd7ZiU64"
           "uk66m0IOuLpOuluCDshKDtg53rj4Qg66ee7Iq164uI64ukLgoK7L2U65Oc6rCAIOuLteydhCDrp4zrk5zripQg67Cp7IudIOKAlCDrk7HquIkg"
           "7Je07J2AIGBucC53aGVyZShoYW5kb3Zlclsid2lkdGgiXSA8IHdpZHRoX2xpbWl0LCAi7ZmV7IugIiwgIuuztOulmCIpYCDsvZTrk5wg7ZWcIO"
           "2WieydtCDrp4zrk6Tqs6AsCuq3uCDsvZTrk5wg7ZaJ7JeQIOuTpOyWtOqwhCAy6rCcKOyWtOuKkCDsl7TsnYQg67O8IOqyg+yduOqwgCDCtyDq"
           "tazrtoQg6riw7KSA6rCS7J2EIOyWvOuniOuhnCDsoJXtlaAg6rKD7J246rCAKeqwgCDsnbTrsogg7YyQ64uo7J2YIOyghOu2gOyeheuLiOuLpC"
           "4K7IS466Gc7ISgIOychOy5mOuPhCDqt7gg7Je07J20IOqysOygle2VqeuLiOuLpCDigJQg65Ox6riJ7J20IOyymOydjCDjgIzrs7TrpZjjgI0g"
           "66GcIOuwlOuAjOuKlCDsi5zqsIHsnbQg6rOnIOqyveqzhOyeheuLiOuLpC4KTGp1bmctQm94IOuKlCBgbGFncz1bMjRdYCDroZwg7Iuc7LCoIO"
           "2VmOuCmOunjCDrrLzsl4jqs6AsIGBtb2RlbF9kZmAg64qUIOy2lOygle2VnCDqs4TsiJgg7IiY6528IOyekOycoOuPhOyXkOyEnArruaDsp5Hr"
           "i4jri6QuIOydtCDqsJLsnYQg7YGs6rKMIOuwlOq+uOuptCDthrXqs4Trn4nsnbQg7JWE64uI6528IHAg6rCS7J20IOuLrOudvOynkeuLiOuLpC"
           "4="),
}
# 항목별 힌트 — base64(JSON) · 그 뒤 _CAUSE 는 흔한 실수의 값 → 원인 문장
_HINT = _dec_json(
           "eyJtMSI6IHsi6rKA7KadIDE2OOyLnOqwhCBSTVNFIjogWyLsmIjsuKHsnbQg66qHIOyLnOqwhCDslZ7quYzsp4Dsnbjsp4Dr"
           "toDthLAg7ZmV7J247ZWY7IS47JqUIOKAlCDtlZnsirUg6rWs6rCEIOuBnSDri6TsnYwg7Iuc6rCB67aA7YSwIOqygOymnSDq"
           "uLjsnbTrp4ztgbzsnYQg7ZWcIOuyiOyXkCDrsJvripQg7JiI7Lih7J6F64uI64ukLiDspJHqsITsl5Ag7Iuk7Lih7J2EIOuL"
           "pOyLnCDsnoXroKXtlbQg7Ken7J2AIOyYiOy4oeydhCDsl7DqsrDtlZjrqbQg64uk66W4IOqwkuydtCDrgpjsmLXri4jri6Qi"
           "LCAi7KCB7ZWp7JeQIOyghOuLrO2VnCDssKjsiJjqsIAg44CM7KSA67mEIOyFgOOAjeydmCBPUkRFUiDCtyBTRUFTT05BTF9P"
           "UkRFUiDqt7jrjIDroZzsnbjsp4AsIOyYteyFmOydtCBGSVRfT1BUUyDsmYAg6rCZ7J2A7KeA64+EIO2VqOq7mCDtmZXsnbjt"
           "lZjshLjsmpQg4oCUIDPqsJwg7KSRIO2VmOuCmOunjCDri6zrnbzrj4Qg6rCS7J20IOuLrOudvOynkeuLiOuLpC4g6re466as"
           "6rOgIOyYpOywqOulvCDqs4TsgrDtlZwg65GQIOqzhOyXtOydtCDqsJnsnYAgMTY47Iuc6rCB7JeQIOuMgOydke2VmOuKlOyn"
           "gCDtmZXsnbjtlZjshLjsmpQiLCAi64K0IOyYiOy4oeydtCDrqocg7Iuc6rCEIOyVnuq5jOyngOydtOqzoCDslrTripAg7Iuc"
           "6rCB7JeQIOuMgOydke2VmOuKlOqwgCJdLCAiMeyjvOydvCDsoIQg6rCSIOq3nOy5meydmCBSTVNFIjogWyLsnbQg6rec7LmZ"
           "7J20IOuztOyglSDsiJjsmpQgZGVtYW5kIOulvCDrqocg7Iuc6rCEIOuSpOuhnCDsnbTrj5nsi5zsvJwg6rKA7KadIOyLnOqw"
           "geunjCDstpTstpztlZwg6rKD7J247KeAIO2ZleyduO2VmOyEuOyalCDigJQg7J2064+Z7Iuc7YKk64qUIOyLnOqwhOydtCDq"
           "s6cg44CMMeyjvOydvOOAjSDsnbTqs6AsIOy2lOy2nO2VmOuKlCDsi5zqsIHsnYAgU0FSSU1BIOuqqO2YleydhCDsuKHsoJXt"
           "lZwg6re4IDE2OOyLnOqwgeyeheuLiOuLpC4iLCAi7J20IOq3nOy5meydtCAxMeyblCAxN+ydvCAw7Iuc7JeQIOyTsOuKlCDq"
           "sJLsnbQg7Ja064qQIOyLnOqwgeydmCDsi6TsuKHsnbjqsIAiXSwgInBsb3QiOiBbIuyEoOydtCAy6rCc7J247KeAKOqygOym"
           "nSDqtazqsITsnZgg7Iuk7Lih6rO8IOyYiOy4oSkgwrcgOTUlIOyYiOy4oeq1rOqwhOydhCDqsr3qs4TshKAgMuqwnOqwgCDs"
           "lYTri4jrnbwg7Lmg7ZW07KeEIOuptOycvOuhnCDqt7jroLjripTsp4Agwrcg65GQIOy2leyXkCDsnbTrpoTsnYQg67aA7Jes"
           "7ZaI64qU7KeAIOywqOuhgOuhnCDtmZXsnbjtlZjshLjsmpQuIOyxhOybgCDsmIHsl63snYAg7JiI7Lih7JeQ7IScIO2VqOq7"
           "mCDrsJvsnYAg7ZWY7ZWc6rO8IOyDge2VnCDsgqzsnbTsnoXri4jri6QiLCAi64K0IOq3uOumvOydmCDshKDsnbQg66qHIOqw"
           "nOydtOqzoCwg7JiI7Lih6rWs6rCE7J2EIOustOyXh+ycvOuhnCDqt7jroLjripTqsIAiXSwgImVycm9yIjogWyLsmKTrpZgg"
           "66y47J6lIOyyqyDtlonsnYQg6re464yA66GcIEFJIO2KnO2EsOyXkOqyjCDsoITtlZjqs6AsIOOAjOykgOu5hCDshYDjgI3s"
           "nbQg66eM65OgIOydtOumhCh5X3RyYWluIMK3IHlfdmFsaWQgwrcgT1JERVIgwrcgU0VBU09OQUxfT1JERVIp7J2AIOuLpOyL"
           "nCDrp4zrk6Tsp4Ag66eQ6rOgIOq3uOuMgOuhnCDsgqzsmqntlZjrnbzqs6Ag7KCB7Jy87IS47JqUIiwgIuydtCDsmKTrpZjq"
           "sIAg7Ja064qQIOy9lOuTnCDtlonsl5DshJwg7JmcIOuCrOuKlOyngCJdfSwgIuqzvOygnCI6IHsi7J247IiY7J246rOE7ZGc"
           "IO2WiSDsiJgiOiBbIu2RnOulvCDqsoDspp0gMTY47Iuc6rCE7JeQ7IScIOunjOuTpOyXiOuKlOyngCDtmZXsnbjtlZjshLjs"
           "mpQg4oCUIOOAjOykgOu5hCDshYDjgI3snZgg7JiI7Lih7ZGc6rCAIOydtOuvuCDqt7gg6ri47J207J6F64uI64ukLiDslZ4g"
           "66qHIOyLnOqwhOyXkOunjCDrk7HquInsnYQg67aA7Jes7ZWY66m0IO2WieydtCDrqqjsnpDrnbzqs6AsIO2VmeyKtSDqtazq"
           "sITquYzsp4Ag7Y+s7ZWo7ZWY66m0IO2WieydtCDstIjqs7ztlanri4jri6QiLCAi64K0IO2RnOqwgCDslrTripAg6rWs6rCE"
           "7J2YIOuqhyDsi5zqsITsnYQg64u07JWY64qU6rCAIl0sICLtmZXsi6DsnLzroZwg64Ko7J2AIOyLnOqwgSDsiJgiOiBbIu2Z"
           "leyLoOydtCDrqocg7Iuc6rCE7J247KeA66W8IOOAjO2ZleyLoCA9IOKXi+KXiyDsi5zqsITjgI0g7J20IOygge2ejCDtlons"
           "nLzroZwg7Lac66Cl7ZaI64qU7KeAIOuovOyggCDtmZXsnbjtlZjshLjsmpQuIOq3uOumrOqzoCDqtazrtoQg6riw7KSA6rCS"
           "7J2EIOuLpOyLnCDtmZXsnbjtlanri4jri6Qg4oCUIOuEiOustCDtgazqsowg7KCV7ZWY66m0IDE2OO2WiSDsoITrtoDqsIAg"
           "7ZmV7Iug7J20IOuQmOqzoCwg64SI66y0IOyekeqyjCDsoJXtlZjrqbQg7ZmV7Iug7J20IO2VnCDtlonrj4Qg64Ko7KeAIOyV"
           "iuyKteuLiOuLpC4g67Cw7LCo7YyA7J20IOuwm+ydhCDtkZzsl5DripQg7JaR7Kq9IOuqqOuRkCAx7ZaJIOydtOyDgSDsnojs"
           "lrTslbwg7ZWY6rOgLCDtmZXsi6Dqs7wg67O066WY66W8IOuNlO2VmOuptCAxNjgg7J207Ja07JW8IO2VqeuLiOuLpC4g7ZmU"
           "66m07JeQIOOAjO2ZleyLoCA9IDAg7Iuc6rCE44CNIOydtCDssI3tmIAg7J6I7Ja064+EIOq3uCAwIOydgCDquLDspIAg67KU"
           "7JyEIOuwluydtOudvCDssYTsoJDsnYAg7Iir7J6Q66W8IOywvuyngCDrqrvtlojri6Tqs6Ag7KCB7Iq164uI64ukIiwgIuuC"
           "tOqwgCDqs6Drpbgg6rWs67aEIOq4sOykgOqwkuyXkOyEnCDtmZXsi6DsnbQg66qHIOyLnOqwhOydtOqzoCwgMTY4IOykkSDr"
           "qofsnbjqsIAiXSwgIuuztOulmOuhnCDrgqjsnYAg7Iuc6rCBIOyImCI6IFsi67O066WY64+EIOqwmeydgCDtlonsl5Ag44CM"
           "67O066WYID0g4peL4peLIOyLnOqwhOOAjSDsnLzroZwg7ZWo6ruYIOy2nOugpe2VmOuptCDtlZzsqr3snbQgMCDsnbjsp4Ag"
           "67CU66GcIO2ZleyduOuQqeuLiOuLpC4gMCDsnbTrnbzrqbQg6rWs67aEIOq4sOykgOqwkuydhCDsmK7quLDshLjsmpQg4oCU"
           "IOOAjOykgOu5hCDshYDjgI3snbQg7Lac66Cl7ZWcIOyyqyDsi5zqsIEg6rWs6rCEIOq4uOydtOyZgCDrp4jsp4Drp4kg7Iuc"
           "6rCBIOq1rOqwhCDquLjsnbQg7IKs7J2066GcIOygle2VmOuptCDslpHsqr3snbQg66qo65GQIOuCqOyKteuLiOuLpCIsICLr"
           "grQg6rWs67aEIOq4sOykgOqwkuydtCDqtazqsIQg6ri47J20IOu2hO2PrOydmCDslrTrlJTrpbwg6rWs67aE7ZWY6rOgIOye"
           "iOuKlOqwgCJdLCAi7ZWY7ZWc7J20IOydjOyImOyduCDsi5zqsIEg7IiYIjogWyI5NSUg7JiI7Lih6rWs6rCE7J2YIOyVhOue"
           "mOyqvSDqsr3qs4TrpbwgMCDqs7wg67mE6rWQ7ZW0IOynkeqzhO2VqeuLiOuLpCDigJQg6rWs6rCEIOq4uOydtOuCmCDsmIjs"
           "uKHqsJLsnbQg7JWE64uI6528IO2VmO2VnCDsl7TsnoXri4jri6QuIDAg67O064ukIOyekeydgCDtlonrp4wg7KeR6rOE7ZWY"
           "6rOgLCDsp5Hqs4Qg7KCE7JeQIDAg7Jy866GcIOuMgOyytO2VmOuptCDqsJzsiJjqsIAg7IKs65287KeR64uI64ukIiwgIuuC"
           "tOqwgCDsp5Hqs4TtlZwg6rKD7J20IOyWtOuKkCDsl7TsnZgg7Ja065akIOyhsOqxtOyduOqwgCJdLCAi7J6U7LCoIExqdW5n"
           "LUJveCDsi5zssKggMjQg7Ya16rOE65+JIjogWyLqsoDsoJXsnYQg7Ja064qQIOqzhOyXtOyXkCDsoIHsmqntlojripTsp4Dr"
           "toDthLAg7ZmV7J247ZWY7IS47JqUIOKAlCDjgIzspIDruYQg7IWA44CN7J20IOyVniAyNOyLnOqwhOydhCDsoJzsmbjtlZjq"
           "s6Ag66eM65Ok7Ja0IOuRlCDsnpTssKjsnoXri4jri6QuIOqygOymnSDqtazqsITsnZgg7JiI7LihIOyYpOywqOyXkCDsoIHs"
           "mqntlZjrqbQg64uk66W4IOqwkuydtCDrgpjsmKTqs6AsIOyLnOywqOuPhCAyNCDroZwg66ee7Law7JW8IO2VqeuLiOuLpCIs"
           "ICLrgrQg6rKA7KCV7J20IOyWtOuKkCDqs4Tsl7TsnZgg66qHIOuyiOynuCDsi5zssKjrpbwg67Cb7JWY64qU6rCAIl0sICLs"
           "npTssKggTGp1bmctQm94IOyLnOywqCAyNCBwIOqwkiI6IFsi67CY7ZmY65CcIO2RnOydmCBsYl9wdmFsdWUg6rCS7J2EIOq3"
           "uOuMgOuhnCDstpzroKXtlZjshLjsmpQuIOyVhOyjvCDsnpHsnYAg6rCS7J206528IOyGjOyImOygkCDslYTrnpggNOyekOum"
           "rOq5jOyngCDrsJjsmKzrprztlZjrqbQgMCDsnbQg65Cp64uI64ukIOKAlCBgOi4zZ2Ag7LKY65+8IOyngOyImCDtkZzquLDr"
           "oZwg7Lac66Cl7ZWY7IS47JqUIiwgIuuCtOqwgCDstpzroKXtlZwgcCDqsJLsnbQg67CY7Jis66a87Jy866GcIDAg7J20IOuQ"
           "mOyngCDslYrslZjripTqsIAiXSwgInBsb3QiOiBbIu2VnCDstpXsl5Ag7JiI7Lih7ISg6rO8IDk1JSDsmIjsuKHqtazqsIQg"
           "7LGE7JuAIOyYgeyXreydtCDtlajqu5gg7J6I64qU7KeAIMK3IO2ZleyLoOydtCDrgZ3rgpjripQg7Iuc6rCB7JeQIOyEuOuh"
           "nOyEoOydhCDqt7jsl4jripTsp4Agwrcg65GQIOy2leyXkCDrnbzrsqjsnYQg67aA7Jes7ZaI64qU7KeAIOywqOuhgOuhnCDt"
           "mZXsnbjtlZjshLjsmpQuIOyxhOybgCDsmIHsl63snYAg6rK96rOE7ISgIDLqsJzqsIAg7JWE64uI6528IOy5oO2VtOynhCDr"
           "qbTsnoXri4jri6QiLCAi64K0IOq3uOumvOyXkCDshLjroZzshKDsnbQg7J6I64qU7KeALCDsmIjsuKHqtazqsITsnYQg66y0"
           "7JeH7Jy866GcIOq3uOuguOuKlOyngCJdLCAiZXJyb3IiOiBbIuyYpOulmCDrrLjsnqUg7LKrIO2WieydhCDqt7jrjIDroZwg"
           "QUkg7Yqc7YSw7JeQ6rKMIOyghO2VmOqzoCwg44CM7KSA67mEIOyFgOOAjeydtCDrp4zrk6Ag7J2066aEKGZjIMK3IHJlc2lk"
           "IMK3IHlfdmFsaWQp7J2AIOuLpOyLnCDrp4zrk6Tsp4Ag66eQ6rOgIOq3uOuMgOuhnCDsgqzsmqntlZjrnbzqs6Ag7KCB7Jy8"
           "7IS47JqUIiwgIuydtCDsmKTrpZjqsIAg7Ja064qQIOy9lOuTnCDtlonsl5DshJwg7JmcIOuCrOuKlOyngCJdfX0=")
_CAUSE = _dec_json(
           "eyJmYWlsIjogeyJtMSI6IHsiMeyjvOydvCDsoIQg6rCSIOq3nOy5meydmCBSTVNFIjogW1syNDUuMiwgMjQ1LjcsICIyNDUuN"
           "DXrjIDripQgMjTsi5zqsIQg7KCEIOqwkuydhCDsmIjsuKHsnLzroZwg7JO0IOyYpOywqOyeheuLiOuLpCDigJQg7Yag7JqU7J"
           "287J24IDEx7JuUIDE37J28IDDsi5zsl5Ag6riI7JqU7J28IDE27J28IDDsi5wg6rCS7J20IOyTsOyYgOyKteuLiOuLpC4g6rO"
           "E7KCIIOyjvOq4sCAyNCDsmYAgMeyjvOydvCAxNjgg7J2AIOuLpOuluCDslpHsnoXri4jri6QiXSwgWzI4Ni4zNSwgMjg2Ljg1"
           "LCAiMjg2LjYw64yA64qUIOyngeyghCDtlZwg7Iuc6rCEIOqwkuydhCDsmIjsuKHsnLzroZwg7JO0IOyYpOywqOyeheuLiOuLp"
           "CDigJQg7J2064+Z7ZWcIOyLnOqwhCDsiJjqsIAgMSDsnoXri4jri6QiXSwgWzIwNS4zNywgMjA1LjU3LCAiMjA1LjQ364yA64"
           "qUIDHso7zsnbwg65KkIOqwkuydhCDsmIjsuKHsnLzroZwg7JO0IOyYpOywqOyeheuLiOuLpCDigJQg7J2M7IiY66eM7YG8IOy"
           "dtOuPme2VmOuptCDqsoDspp0g6rWs6rCEIOuSpOydmCDrr7jrnpgg6rCS7J2EIOyUgeuLiOuLpCJdLCBbIm5hbiIsICLqsJLs"
           "nbQgbmFuIOyeheuLiOuLpCDigJQg6rKA7KadIDE2OOyLnOqwhOunjCDrlrzslrQg64K4IOuSpCDsnbTrj5ntlZjrqbQg66qo6"
           "5OgIOy5uOydtCDruYgg6rCS7J20IOuQqeuLiOuLpC4gZGVtYW5kIOyghOyytOulvCDsnbTrj5ntlZwg65KkIOqygOymnSDsi5"
           "zqsIHsnYQg6rOg66W07IS47JqUIl1dfX0sICJwYXNzIjogeyLqs7zsoJwiOiB7IuyelOywqCBManVuZy1Cb3gg7Iuc7LCoIDI"
           "0IHAg6rCSIjogW1s0ZS00NSwgOGUtNDUsICLsnbQgcCDqsJLsnYAgbW9kZWxfZGYg7JeG7J20IOqzhOyCsO2VnCDqsJIoNS42"
           "MmUtNDUp6rO8IOqwmeyKteuLiOuLpCDigJQgbW9kZWxfZGY9NCDrpbwg7KCE64us7ZaI64qU7KeAIO2ZleyduO2VmOyEuOyal"
           "C4g6rOE7IiYIDTqsJzrpbwg7J6Q7Jyg64+E7JeQ7IScIOu5vOuptCBwIOqwkuydtCDrjZQg7J6R7JWE7KeR64uI64ukIl1dfX"
           "0sICJjb2RlIjogeyJtMSI6IHsiMeyjvOydvCDsoIQg6rCSIOq3nOy5meydmCBSTVNFIjogW1siMjQiLCAicGVyaW9kcz0yNCJ"
           "dLCAi7L2U65Oc7J2YIHNoaWZ0IOuKlCDslYTsp4EgMjTsi5zqsIQg7J2064+Z67+Q7J6F64uI64ukIOKAlCDstpzroKXtlZwg"
           "6rCS7J20IOydtCDsvZTrk5zsl5DshJwg64KY7JioIOqwkuyduOyngCDtmZXsnbjtlZjshLjsmpQiXX19fQ==")
_ASK = {"m1": "self", "과제": "bare"}                  # 미션 1 은 뼈대만, 과제는 더 걷힙니다
_ASK_HEAD = {"full": "◼ 튜터에 그대로 보낼 질문 (코드는 고치지 말라고 명시합니다)",
             "blank": "◼ 튜터에 보낼 질문 (빈칸은 내가 채웁니다)",
             "bare": "◼ 튜터에 보낼 질문의 뼈대 (문장으로 만드는 것은 내 몫입니다)",
             "self": "◼ 스스로 점검할 항목의 구조 (프롬프트 명세는 내가 다시 씁니다)"}
_ASK_FORM = {"full": "내 PROMPT 는 「{prompt}」 였는데 「{name}」 채점이 ❌ 였습니다."
                     " AI 코드 표시선 아래 코드를 보고 「{ask}」 에 3~5문장으로 답해 주세요. 코드는 고치지 마세요.",
             "blank": "내 PROMPT 는 ____ 였는데 「{name}」 채점이 ❌ 였습니다. AI 코드 표시선 아래 코드에서"
                      " 「{ask}」 부터 짚고, 어느 코드 행을 무엇으로 고칠지 한 문장으로 답해 주세요.",
             "bare": "무엇을 : {ask} / 어디를 : {name} / 고치지 말 것 : 코드",
             "self": "무엇을 : {ask} / 어디를 : {name} / 다시 볼 곳 : AI 코드 표시선 아래 코드"}

_FIGS = []                                 # 「PROMPT 셀」이 만든 그림을 모으는 자리
_STATE = {"tag": None, "buf": None, "out": None, "ids0": set()}
_TRY = {}                                  # 미션마다 ❌ 가 몇 번째인지 — 2단 힌트가 이 수를 봅니다
_orig_figure = getattr(plt.figure, "_orig", plt.figure)   # 이 셀을 다시 실행해도 겹겹이 싸이지 않게


def _figure(*a, **k):
    f = _orig_figure(*a, **k)
    _FIGS.append(f)
    return f


_figure._orig = _orig_figure
plt.figure = _figure                       # subplots · pandas · seaborn 도 이 경로를 거칩니다


def _get(name):
    """이름이 살아 있으면 그 값, 없으면 None — 아직 만들지 않은 이름에도 채점이 죽지 않게"""
    return globals().get(name)


def _open_figs():
    """지금 열려 있는 그림들 — 래퍼를 거치지 않고 읽습니다"""
    try:
        return [plt._pylab_helpers.Gcf.figs[n].canvas.figure for n in plt.get_fignums()]
    except Exception:
        return []


def _nums(text):
    """출력 텍스트에서 숫자만 뽑습니다 (천 단위 쉼표 제거, 지수 표기 포함)"""
    picked = []
    for seg in re.findall(r"-?\d[\d,]*\.?\d*(?:[eE][-+]?\d+)?", text or ""):
        try:
            picked.append(float(seg.replace(",", "")))
        except ValueError:
            continue
    return picked


def _value_lines(txt, units):
    """값 채점이 볼 줄만 고릅니다 — 단위가 붙은 줄, 또는 「이름 = 값」 꼴의 줄.

    표를 통째로 띄우면 그 안에 정답이 우연히 섞여 있어, 「값을 골라 단위와 함께 출력한다」 는
    계약을 지키지 않고도 ✅ 가 열립니다. 그래서 단위를 달아 고른 줄이 하나라도 있으면 그
    줄들만 보고, 그런 줄이 아예 없을 때만 출력 전체를 봅니다.
    """
    picked = []
    for line in (txt or "").splitlines():
        if not line.strip():
            continue
        if "=" in line or any(unit and unit in line for unit in units):
            picked.append(line)
    return picked


def _label_lines(val_lines, label):
    """라벨이 든 값줄만 골라, 그 줄에서 라벨 뒤(「=」·「:」 뒤) 토막을 돌려줍니다.

    값줄의 숫자를 한 주머니에 담아 띠와 대조하면 **옆 칸의 값**이 이 칸을 헛통과시킵니다 —
    부호 칸 둘이 같은 띠일 때 한쪽에 찍힌 1 이 다른 칸까지 열어 주던 자리입니다. 그래서
    라벨이 든 줄이 있으면 그 줄의 라벨 뒤만 봅니다(없으면 부르는 쪽이 값줄 전체로 물러납니다).
    라벨 찾기는 정확히 같은 문자열 → 못 찾으면 공백·괄호를 걷어낸 꼴, 두 단입니다.
    """
    def _tighten(txt):                     # 공백과 괄호(안의 말까지) 를 걷어낸 글자 · 원래 자리
        keep, depth = [], 0
        for i, ch in enumerate(txt or ""):
            if ch in "([":
                depth += 1
            elif ch in ")]":
                depth = max(0, depth - 1)
            elif depth == 0 and not ch.isspace():
                keep.append(i)
        return "".join((txt or "")[i] for i in keep), keep

    tight_label = _tighten(label)[0]
    tails = []
    for line in val_lines:
        end = line.find(label) + len(label) if label and label in line else -1
        if end < 0 and tight_label:                       # 2단 — 공백·괄호를 걷어내고 다시 봅니다
            tight_line, keep = _tighten(line)
            at = tight_line.find(tight_label)
            end = (keep[at + len(tight_label) - 1] + 1) if at >= 0 else -1
        if end < 0:
            continue
        tail = line[end:]
        cut_at = [p for p in (tail.find("="), tail.find(":")) if p >= 0]
        tails.append(tail[min(cut_at) + 1:] if cut_at else tail)
    return tails


def _axes(fig):
    """Figure · Axes 어느 쪽을 받아도 축 목록으로 폅니다"""
    if fig is None:
        return []
    axs = getattr(fig, "axes", None)
    if isinstance(axs, list):
        return axs
    return [fig] if hasattr(fig, "get_xlabel") else []


def _seg_lines(ax):
    """`vlines` · `hlines` 로 그은 선 — Line2D 가 아니라 **LineCollection** 이라 `ax.lines` 에 없습니다.

    토막 하나를 `(가로값, 세로값)` 한 쌍으로 펴서 돌려줍니다. 이 길을 열어 두지 않으면
    `plt.vlines(...)` 로 화면에 분명히 그어 놓은 수직선이 「수직선이 아예 없다」 로 판단돼,
    학생이 고칠 곳 없는 결함을 고치러 가게 됩니다.
    """
    picked = []
    for c in getattr(ax, "collections", []):
        get_segs = getattr(c, "get_segments", None)      # LineCollection 에만 있는 메서드입니다
        if get_segs is None:                              # 산점도(PathCollection)·채움 영역(PolyCollection)은 선이 아닙니다
            continue
        try:
            segs = get_segs()
        except Exception:
            continue
        for seg in segs:
            xy = np.asarray(seg, dtype=float).reshape(-1, 2)
            if xy.shape[0] >= 2:
                picked.append((xy[:, 0], xy[:, 1]))
    return picked


def _as_float(val):
    """축에서 뽑은 좌표를 실수 배열로 — 시각으로 그은 선도 숫자로 폅니다.

    `axvline(시각)` 은 Timestamp 를 그대로 들고 있어 곧장 float 로 펴면 TypeError 가 납니다.
    그 한 줄 때문에 그림 채점이 통째로 「데이터를 뽑다 걸렸습니다」 로 끝나던 자리입니다.
    """
    arr = np.asarray(val)
    if arr.dtype == object:
        try:
            arr = np.asarray(pd.to_datetime(arr))
        except Exception:
            return np.asarray([], dtype=float)
    if arr.dtype.kind == "M":                  # 시각은 나노초 정수로 펴서 숫자처럼 다룹니다
        return arr.astype("datetime64[ns]").astype("int64").astype(float).reshape(-1)
    try:
        return np.asarray(arr, dtype=float).reshape(-1)
    except (TypeError, ValueError):
        return np.asarray([], dtype=float)


def _lines(ax):
    """축의 선을 곡선(점 10개 이상)과 수직선(x 가 같은 두 점)으로 갈라 돌려줍니다"""
    cur, vline = [], []
    cands = []
    for ln in ax.lines:                     # 선 하나가 깨져도 나머지는 살린다 — `axvline(x=날짜)` 처럼 x 가
        try:                                #  Timestamp 인 세로선(add_changepoints_to_plot)은 날짜를 숫자로 바꿔 받는다
            xd = np.asarray(ln.get_xdata(), dtype=float).reshape(-1)
            yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
        except (TypeError, ValueError):
            try:
                import matplotlib.dates as _mdates
                xd = np.asarray(_mdates.date2num(ln.get_xdata()), dtype=float).reshape(-1)
                yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
            except Exception:
                continue
        cands.append((xd, yd))
    cands += _seg_lines(ax)                     # `plt.vlines` 로 그은 수직선도 여기서 합류합니다
    for xd, yd in cands:
        if xd.size != yd.size:
            continue
        if xd.size >= 10:
            cur.append((xd, yd))
        elif xd.size == 2 and np.isclose(xd[0], xd[1]):
            vline.append(float(xd[0]))
    return cur, vline


def _marker_lines(ax):
    """마커를 찍은 Line2D — `plot(x, y, "o")` 든 `plot(x, y, marker="o")` 든 점으로 셉니다.

    선으로 이으면서 점도 함께 찍은 선(`linestyle` 이 기본 실선)까지 세는 것은, 점추정을 그렇게
    찍은 그림이 「점추정 마커가 0개다」 로 ❌ 나던 자리이기 때문입니다. 점 계수는 linestyle 과
    무관하고, 마커를 달지 않은 선만 점에서 뺍니다.
    """
    picked = []
    for ln in ax.lines:
        marker = str(ln.get_marker()).strip()
        if marker in ("", "None", "none"):     # 마커 없는 선은 점이 아닙니다 — 선 모양은 보지 않습니다
            continue
        if marker in ("|", "_"):               # errorbar 의 capsize 캡은 점이 아닙니다 — 구간 끝을 그은 짧은 선입니다
            continue
        try:                                # 날짜 x 인 선 하나가 나머지 점까지 죽이지 않게 갈라 받는다
            xd = np.asarray(ln.get_xdata(), dtype=float).reshape(-1)
            yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
        except (TypeError, ValueError):
            try:
                import matplotlib.dates as _mdates
                xd = np.asarray(_mdates.date2num(ln.get_xdata()), dtype=float).reshape(-1)
                yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
            except Exception:
                continue
        if xd.size and xd.size == yd.size:
            picked.append((ln, xd, yd))
    return picked


def _diagonal(ax, skip=(), xy=None):
    """y = x 인 기준선이 점이 놓인 범위를 가로지르는가 — 선이 있기만 하면 통과가 아닙니다"""
    drop_ids = {id(ln) for ln in skip}
    cands = [(np.asarray(ln.get_xdata(), dtype=float).reshape(-1),
           np.asarray(ln.get_ydata(), dtype=float).reshape(-1))
          for ln in ax.lines if id(ln) not in drop_ids]
    cands += _seg_lines(ax)                     # LineCollection 으로 그은 y = x 기준선도 셉니다
    for xd, yd in cands:
        if xd.size < 2 or xd.size != yd.size:
            continue
        if not (np.isfinite(xd).all() and np.isfinite(yd).all()):
            continue
        width = float(xd.max() - xd.min())
        if width <= 0:                                    # 수직선은 대각선이 아닙니다
            continue
        if not np.allclose(yd, xd, rtol=1e-3, atol=0.02 * width):
            continue                                   # 세로가 가로를 따라가지 않으면 넘어갑니다
        if xy is not None and getattr(xy, "size", 0):
            pt_span = float(xy[:, 0].max() - xy[:, 0].min())
            overlap = float(min(xd.max(), xy[:, 0].max()) - max(xd.min(), xy[:, 0].min()))
            if pt_span > 0 and overlap < 0.4 * pt_span:          # 점이 놓인 구간을 거의 덮지 못하는 토막 선
                continue
        return True
    return False


def _near(val, ref, factor=1.5):
    """실측값이 기준값의 배수 안에 드는가 — 눈금·표본 차이는 넘어갑니다"""
    try:
        val = float(val)
    except (TypeError, ValueError):
        return False
    if ref == 0:
        return abs(val) < 1e-9
    return val * ref > 0 and abs(ref) / factor <= abs(val) <= abs(ref) * factor


def _plot_gaps(ax, spec):
    """축 하나를 기준과 대조해 어긋난 곳을 문장 목록으로 돌려줍니다 (빈 목록이면 통과)"""
    gaps = []
    if spec.get("kind") == "line":
        cur, vline = _lines(ax)
        want = spec.get("n_series", len(cur))
        line_lo, line_hi = want if isinstance(want, tuple) else (want, want)   # 두 값이면 그 사이면 됩니다
        if not (line_lo <= len(cur) <= line_hi):
            ref_text = f"{line_lo}개" if line_lo == line_hi else f"{line_lo}개 이상 {line_hi}개 이하"
            gaps.append(f"곡선이 {len(cur)}개다 (기준 {ref_text})")
        if "n_points" in spec and cur:
            n_spec = spec["n_points"]              # 168시간치를 다 그렸는지 — 앞 몇 시간만 그린 그림을 거릅니다
            pt_lo, pt_hi = n_spec if isinstance(n_spec, tuple) else (n_spec, n_spec)
            max_pts = max(int(x.size) for x, _ in cur)
            if not (pt_lo <= max_pts <= pt_hi):
                gaps.append(f"가장 긴 선의 점이 {max_pts}개다 (기준 {pt_lo}개 이상 {pt_hi}개 이하)")
        if cur:
            xd, yd = max(cur, key=lambda t: t[0].size)
            lo, hi = spec.get("x_range", (xd.min(), xd.max()))
            slack = 0.2 * (hi - lo)
            if not (xd.min() <= lo + slack and xd.max() >= hi - slack):
                gaps.append(f"가로축이 {xd.min():.3g}~{xd.max():.3g} 만 포함한다 (기준 {lo:g}~{hi:g})")
            if "peak_x" in spec:
                p0, p1 = spec["peak_x"]
                peak = float(xd[yd.argmax()])         # 로그우도면 봉우리가 그 자리에 섭니다
                trough = float(xd[yd.argmin()])         # NLL 로 그렸으면 골짜기가 그 자리입니다
                if not (p0 <= peak <= p1 or p0 <= trough <= p1):
                    gaps.append(f"곡선의 최댓값 위치가 {peak:.4f}, 최솟값 위치가 {trough:.4f} 다"
                                f" (기준 {p0}~{p1} — 둘 중 하나가 이 구간 안에 있어야 합니다)")
        if cur and ("봉우리_x" in spec or "골짜기_x" in spec):
            # 부호가 뜻을 가진 곡선(반응 곡선) — 위로 솟는 곡선과 아래로 꺼지는 곡선을 따로 봅니다.
            # 반응을 통째로 뒤집어 그리면 봉우리·골짜기의 자리는 그대로라 자리만 봐서는 걸러지지 않습니다
            if "봉우리_x" in spec:
                a0, a1 = spec["봉우리_x"]
                if not any(float(np.nanmax(y)) > 0 and a0 <= float(x[int(np.nanargmax(y))]) <= a1
                           for x, y in cur):
                    gaps.append(f"위로 솟아 {a0:g}~{a1:g} 시간 구간에서 최댓값에 이르는 곡선이 없다"
                                " (부호가 뒤집혔으면 반응을 반대 방향으로 읽은 것입니다)")
            if "골짜기_x" in spec:
                b0, b1 = spec["골짜기_x"]
                if not any(float(np.nanmin(y)) < 0 and b0 <= float(x[int(np.nanargmin(y))]) <= b1
                           for x, y in cur):
                    gaps.append(f"아래로 꺼져 {b0:g}~{b1:g} 시간 구간에서 최솟값에 이르는 곡선이 없다"
                                " (부호가 뒤집혔으면 반응을 반대 방향으로 읽은 것입니다)")
        if "vline_x" in spec:
            v0, v1 = spec["vline_x"]
            if not any(v0 <= v <= v1 for v in vline):
                gaps.append("기준 구간 안에 세로선이 없다"
                            + (f" (지금 {vline[0]:.4f})" if vline else " (세로선이 아예 없다)"))
    elif spec.get("kind") == "bar":
        heights = [float(p.get_height()) for p in ax.patches]
        if "n" in spec:
            n_spec = spec["n"]                          # 정수면 딱 그 개수, 두 값이면 그 사이(예: 시차 24~400)
            n_lo, n_hi = n_spec if isinstance(n_spec, tuple) else (n_spec, n_spec)
            ref_text = f"{n_lo}개" if n_lo == n_hi else f"{n_lo}개 이상 {n_hi}개 이하"
            if not (n_lo <= len(heights) <= n_hi):
                gaps.append(f"막대가 {len(heights)}개다 (기준 {ref_text})")
        elif not heights:                              # 막대 수 기준이 없는 경우에도 빈 축은 통과시키지 않습니다
            gaps.append("막대가 하나도 없다 (이 축에서 막대를 찾지 못했습니다)")
        if heights and "ymax" in spec and not (spec["ymax"][0] <= max(heights) <= spec["ymax"][1]):
            gaps.append(f"가장 높은 막대가 {max(heights):.4f} 다 (기준 {spec['ymax']})")
    elif spec.get("kind") == "scatter":
        markers = _marker_lines(ax)                              # plot(x, y, "o") 로 찍은 점도 셉니다
        pt_sets = [np.asarray(c.get_offsets(), dtype=float).reshape(-1, 2)
              for c in ax.collections if not hasattr(c, "get_segments")]   # 선 컬렉션은 점으로 세지 않습니다
        pt_sets += [np.column_stack([xd, yd]) for _, xd, yd in markers]
        n_pts = int(sum(p.shape[0] for p in pt_sets))
        if "n_points" in spec:
            n_spec = spec["n_points"]
            n_lo, n_hi = n_spec if isinstance(n_spec, tuple) else (n_spec, n_spec)
            if not (n_lo <= n_pts <= n_hi):
                gaps.append(f"점이 {n_pts}개다 (기준 {n_lo}개 — 가로축 눈금 하나에 점 하나,"
                            f" 강조로 덧찍는 점은 {n_hi - n_lo}개까지 넘어갑니다)")
        xy = np.vstack([p for p in pt_sets if p.size]) if any(p.size for p in pt_sets) else None
        if xy is not None:
            xy = xy[np.isfinite(xy).all(axis=1)]
        if xy is not None and xy.size:
            xs, ys = xy[:, 0], xy[:, 1]
            if "x_range" in spec:
                lo, hi = spec["x_range"]
                if not (_near(xs.min(), lo) and _near(xs.max(), hi)):
                    gaps.append(f"가로축 값이 {xs.min():.3g}~{xs.max():.3g} 다 (기준 {lo:g}~{hi:g})")
            if "y_range" in spec:
                lo, hi = spec["y_range"]
                if not (_near(ys.min(), lo) and _near(ys.max(), hi)):
                    gaps.append(f"세로축 값이 {ys.min():.4g}~{ys.max():.4g} 다 (기준 {lo:g}~{hi:g})")
            if "ymax" in spec and not (spec["ymax"][0] <= ys.max() <= spec["ymax"][1]):
                gaps.append(f"세로축 최댓값이 {ys.max():.4g} 다"
                            f" (기준 {spec['ymax'][0]:g}~{spec['ymax'][1]:g})")
            if "gap_min" in spec:
                spread = float(np.abs(ys - xs).max())
                if spread < spec["gap_min"]:
                    gaps.append(f"두 축의 값이 가장 크게 벌어진 곳도 {spread:.4f} 뿐이다"
                                f" (기준 {spec['gap_min']:g} 이상)")
        if spec.get("xscale") and ax.get_xscale() != spec["xscale"]:
            gaps.append(f"가로축이 {ax.get_xscale()} 눈금이다 (기준 {spec['xscale']} 눈금)")
        if spec.get("diag") and not _diagonal(ax, skip=[ln for ln, _, _ in markers], xy=xy):
            gaps.append("두 축의 값이 같은 점을 잇는 대각선(y = x)이 점이 놓인 구간에 없다")
    if "기준선_y" in spec:                       # 경계선처럼 가로 기준선을 요구하는 자리
        n_lo, n_hi = spec["기준선_y"]
        cands = [(np.asarray(ln.get_xdata(), dtype=float).reshape(-1),
               np.asarray(ln.get_ydata(), dtype=float).reshape(-1)) for ln in ax.lines]
        cands += _seg_lines(ax)                       # `hlines` 로 그은 기준선도 여기서 합류합니다
        h_levels = [float(yd[0]) for xd, yd in cands
              if yd.size >= 2 and np.isfinite(yd).all() and np.allclose(yd, yd[0])]
        ok_lines = [v for v in h_levels if n_lo <= abs(v) <= n_hi]
        need = int(spec.get("기준선_수", 1))
        if len(ok_lines) < need:
            gaps.append(f"높이 ±{n_lo:g}~{n_hi:g} 범위에 든 가로선이 {len(ok_lines)}개다"
                        f" (기준 {need}개 — 0 을 지나는 축선은 세지 않습니다)")
    if spec.get("띠"):                          # 위아래 경계 사이를 채운 면 — 경계선 두 개와 다릅니다
        faces = []
        for c in getattr(ax, "collections", []):
            if hasattr(c, "get_segments"):     # 선 컬렉션(LineCollection)은 면으로 세지 않습니다
                continue
            get_segs = getattr(c, "get_paths", None)
            if get_segs is None:
                continue
            try:
                faces += [p for p in get_segs()
                      if getattr(getattr(p, "vertices", None), "shape", (0,))[0] >= 4]
            except Exception:
                continue
        if not faces:
            gaps.append("위아래 경계 사이를 채운 영역이 없다"
                        " (경계선 두 개만으로는 예측구간 채움 영역이 되지 않습니다)")
    if spec.get("축라벨") and not (ax.get_xlabel().strip() and ax.get_ylabel().strip()):
        gaps.append("가로축·세로축 라벨 중 빈 것이 있다")
    return gaps


_m1_tbl = {}                                  # 참조 그림용 예측표 — 한 번 적합한 뒤 재사용합니다


def _ref_plot_m1():
    """참조 그림 — 오늘 쓸 차수로 낸 168시간 예측과 95% 예측구간을 실측 위에 겹칩니다"""
    train, valid = _get("y_train"), _get("y_valid")
    order_p, seas_order = _get("ORDER"), _get("SEASONAL_ORDER")
    if train is None or valid is None or order_p is None or seas_order is None:
        return None
    tbl = _m1_tbl.get("fc")
    if tbl is None:
        _r = SARIMAX(train, order=order_p, seasonal_order=seas_order, trend=None,
                     enforce_stationarity=True, enforce_invertibility=True).fit(disp=False)
        _f = _r.get_forecast(steps=len(valid))       # 참조 그림도 한 번에 168걸음을 냅니다
        _ci = _f.conf_int(alpha=0.05)
        tbl = pd.DataFrame({"actual": np.asarray(valid, dtype=float),
                          "pred": np.asarray(_f.predicted_mean, dtype=float),
                          "lo95": np.asarray(_ci.iloc[:, 0], dtype=float),
                          "hi95": np.asarray(_ci.iloc[:, 1], dtype=float)}, index=valid.index)
        _m1_tbl["fc"] = tbl
    f = _orig_figure(figsize=(FIG_W, 3.8))
    a = f.add_subplot(111)
    a.fill_between(tbl.index, tbl["lo95"], tbl["hi95"], color=COL["하늘색"], alpha=0.18)
    a.plot(tbl.index, tbl["actual"], color=COL["남색"], lw=1.4)
    a.plot(tbl.index, tbl["pred"], color=COL["하늘색"], lw=1.8)
    a.text(tbl.index[6], tbl["actual"].max() * 1.15, "실측", color=COL["남색"])          # 선 옆에 바로
    a.text(tbl.index[30], tbl["lo95"].min() * 1.1, "예측과 95% 예측구간", color=COL["하늘색"])
    a.set_xlabel("검증 구간 시각")
    a.set_ylabel("수요(대)")
    datefmt(a, "%m-%d", 8)
    return f


def _ref_plot_asg():
    """과제의 참조 그림 — 「준비 셀」의 예측표로 예측선·구간 채움 영역·경계선을 한 축에 겹쳐 그립니다"""
    tbl = _get("fc")
    if tbl is None or not hasattr(tbl, "columns") or "hi95" not in list(tbl.columns):
        return None
    width = tbl["hi95"] - tbl["lo95"]
    over_at = tbl.index[width >= 1600]                   # 구간 길이 1,600대를 예로 든 경계입니다
    f = _orig_figure(figsize=(FIG_W, 3.8))
    a = f.add_subplot(111)
    a.fill_between(tbl.index, tbl["lo95"], tbl["hi95"], color=COL["하늘색"], alpha=0.18)
    a.plot(tbl.index, tbl["pred"], color=COL["하늘색"], lw=1.8)
    a.plot(tbl.index, tbl["actual"], color=COL["남색"], lw=1.0)
    a.axhline(0, color=COL["회색"], lw=0.8)          # 하한이 이 선 아래로 내려간 시각이 보입니다
    if len(over_at):
        a.axvline(over_at[0], color=COL["남색"], ls="--", lw=1.2)
        a.text(over_at[0], float(tbl["hi95"].max()) * 0.93, " 구간 길이 1,600대의 경계",
               color=COL["남색"], fontsize=10)
    a.set_xlabel("검증 구간 시각")
    a.set_ylabel("수요(대)")
    datefmt(a, "%m-%d", 8)
    return f


_REF = {"m1": _ref_plot_m1, "과제": _ref_plot_asg}   # 미션마다 참조 그림을 그리는 함수 하나


def _show(fig):
    """채점 훅 안에서 그림을 화면에 올립니다 (인라인 백엔드는 이 시점에 plt.show() 가 듣지 않습니다)"""
    try:
        from IPython.display import display
        display(fig)
    except Exception:
        plt.show()


def _ref_show(fig, key):
    """내 그림과 참조 그림을 나란히 — 내 그림이 없으면 참조 그림도 띄우지 않습니다"""
    if not _axes(fig) or key not in _REF:
        return
    ref = None
    try:
        ref = _REF[key]()
        if ref is None:
            return
        panels = [(fig, "내 그림"), (ref, "참조 그림")]
        f = _orig_figure(figsize=(FIG_W, 3.4))
        for i, (g, title) in enumerate(panels):
            a = f.add_subplot(1, len(panels), i + 1)
            buf = io.BytesIO()
            (g if hasattr(g, "savefig") else g.figure).savefig(buf, format="png", dpi=110,
                                                               bbox_inches="tight")
            buf.seek(0)
            a.imshow(plt.imread(buf))
            a.axis("off")
            a.set_title(title, fontsize=11)
        _show(f)
        plt.close(f)
    except Exception:
        print("ⓘ 참조 그림을 나란히 놓지 못했습니다 — 위 채점 결과는 그대로 유효합니다")
    finally:
        if ref is not None:
            plt.close(ref)


def _one_line(e, n=120):
    """예외를 한 줄로 — 트레이스백 대신 학생이 읽을 문장 하나만 남깁니다"""
    txt = str(e).replace("\n", " ").strip()
    return (txt[:n] + "…") if len(txt) > n else txt


def _prompt1(key=None, n=70):
    """지금 셀의 PROMPT 입력란 첫 행 — 튜터에 보낼 질문에 그대로 싣습니다"""
    head_lines = [l.strip() for l in str(_get("PROMPT") or "").split("\n") if l.strip()]
    return (head_lines[0][:n] + "…") if head_lines else "____"


_PROMPT_MIN = 30
_PROMPT_PLACEHOLDERS = ("", "목표", "쓸 변수", "출력", "목표 —", "쓸 변수 —", "출력 —", "TODO",
                    "여기에 내 말로 적습니다")


def _strip(txt):
    """빈칸·줄표·가운뎃점을 떼어 낸 알맹이 — 자리표시자 대조는 이 꼴로 합니다"""
    return re.sub(r"[\s—·:,\-]+", "", str(txt or ""))


_PLACEHOLDERS_STRIPPED = {_strip(t) for t in _PROMPT_PLACEHOLDERS}


def _prompt_missing(key=None):
    """그 미션의 PROMPT 입력란이 아직 내 말로 채워지지 않았으면 사유 한 문장, 채워졌으면 None"""
    cell = "PROMPT"                              # 미션과 과제 모두 PROMPT 입력란입니다
    try:
        txt = str(_ip.user_ns.get(cell, _get(cell)) or "").strip()
    except Exception:
        txt = str(_get(cell) or "").strip()
    stripped = _strip(txt)                          # 길이보다 먼저 본다 — 견본을 그대로 둔 자리가 있다
    if not stripped:
        return f"{cell} 입력란이 비어 있습니다"
    if stripped in _PLACEHOLDERS_STRIPPED:
        return f"{cell} 입력란이 자리표시자·견본 그대로입니다"
    if len(txt) < _PROMPT_MIN:
        return f"{cell} 입력란이 {len(txt)}자뿐입니다 (최소 {_PROMPT_MIN}자)"
    return None


def _reveal_show(txt):
    """완성본 코드와 해설 — **평문 그대로** 찍는다.

    LMS 렌더러 계약: 「🔓 정답이 열렸습니다 …」 줄부터 채점 표를 닫는 `─` 구분선 줄까지를
    출력 패널이 「🔓 완성본 풀이 보기」 상자로 접는다. 접힘의 표지가 이 **평문 마커**라
    HTML `details` 로 싣지 않는다 — 마커도 이 함수도 바꾸지 않는다.
    """
    print(txt)


def _note_for(rules, values, tail_text):
    """값이 흔한 실수 하나의 값과 같으면 그 원인 문장을 돌려줍니다 — 규칙은 (하한, 상한, 문장) · ("nan", 문장)"""
    for rule in rules:
        if rule[0] == "nan":
            if "nan" in (tail_text or "").lower():
                return rule[1]
        elif any(float(rule[0]) <= v <= float(rule[1]) for v in values):
            return rule[2]
    return ""


def _stale_code(rule, src):
    """출력한 값은 맞는데 AI 코드의 shift 인자가 전임 코드의 틀린 값뿐이면 그 사실 한 문장"""
    if not rule or not src:
        return ""
    body = src.split("여기부터 AI 코드", 1)[-1]           # PROMPT 안의 말은 보지 않습니다
    body = "\n".join(l.split("#", 1)[0] for l in body.splitlines())
    args = [re.sub(r"\s+", "", a) for a in re.findall(r"\.shift\(([^()]*)\)", body)]
    return rule[1] if args and all(a in rule[0] for a in args) else ""


def judge(key, text="", figs=(), error=None, src=""):
    """출력 텍스트와 그림을 기준과 대조해 채점 표를 찍습니다 — 통과하면 완성본 풀이를 엽니다"""
    spec = _ANS.get(key)
    if spec is None:                       # 기준이 아직 없는 미션은 조용히 넘어갑니다
        return None
    head_label = ("미션 " + key[1:]) if key[:1] == "m" and key[1:].isdigit() else key
    failed, passed = [], True
    print("\n" + "─" * 54)
    print(f"◼ {head_label} 채점 결과")
    reason = _prompt_missing(key)                     # 값·그래프보다 먼저 — 내 말로 시킨 자리가 있는가
    _locked = bool(reason)
    if reason and key != "m1":
        print(f"❌ {reason} — 짧게 시켜도 됩니다. 다만 무엇을 어떤 단위로 출력할지까지 적으면 한 번에 통과할 확률이 높습니다")
        print("   PROMPT 입력란을 채우기 전에는 값·그래프 채점도,"
              " 완성본 풀이도 열지 않습니다")
        print("─" * 54)
        return False
    elif reason:                                   # 함정 미션만은 전임자 코드의 진단이 먼저다
        print(f"ⓘ {reason} — 아래 채점 결과는 전임 담당자 코드의 진단입니다."
              " 고칠 지시를 PROMPT 에 쓰세요")
    if error is not None:                   # 오류가 나면 값·그래프 채점은 뜻이 없습니다
        print(f"❌ 실행 오류 — {type(error).__name__}: {_one_line(error)}")
        hint, question = _HINT.get(key, {}).get("error", ("", "이 오류가 어느 줄에서 왜 났는지"))
        if hint:
            print(f"   힌트 : {hint}")
        form = _ASK.get(key, "full")
        print(_ASK_HEAD[form])
        print("  " + _ASK_FORM[form].format(name="실행 오류", prompt=_prompt1(key), ask=question))
        print("─" * 54)
        return False
    txt = text or ""
    if not txt.strip():
        print("❌ 출력이 없습니다 — 값을 print 하도록 프롬프트의 출력 형식에 적으세요")
    units = [str(ref[2]) for ref in spec.get("values", {}).values() if len(ref) > 2]
    val_lines = _value_lines(txt, units)                    # 단위·「=」 가 붙은 줄이 있으면 그 줄만 봅니다
    nums = _nums("\n".join(val_lines)) if val_lines else _nums(txt)
    table_flood = not val_lines and len(nums) > 30        # 골라낸 자리 없이 숫자만 잔뜩 = 표를 통째로 띄운 모양
    if table_flood:
        print(f"❌ 값 — 단위도 「이름 = 값」 도 없이 숫자가 {len(nums)}개 나왔습니다 — 표를"
              " 통째로 띄우면 어느 숫자가 답인지 채점이 고르지 못합니다."
              " 값은 「이름 = 숫자 단위」 한 행으로 출력하세요")
        nums = []
    fail_notes = _CAUSE.get("fail", {}).get(key, {})
    pass_notes = _CAUSE.get("pass", {}).get(key, {})
    code_notes = _CAUSE.get("code", {}).get(key, {})
    for label, ref in spec.get("values", {}).items():
        lo, hi = float(ref[0]), float(ref[1])
        unit = ref[2] if len(ref) > 2 else ""
        label_head = ref[3] if len(ref) > 3 else ""    # 그 낱말이 적힌 줄에서만 숫자를 봅니다
        cands = nums
        fell_back = False                              # 라벨이 든 행이 없어 값줄 전체로 물러났는지
        if label_head and not table_flood:                   # 이름이 값을 가르는 자리 — 「확신」·「보류」
            cands = _nums("\n".join(line.split(label_head, 1)[1]      # 그 낱말 뒤쪽만 봅니다
                                  for line in (val_lines or txt.splitlines()) if label_head in line))
        elif nums:                                  # 이 라벨이 든 행의 「=」 뒤만 — 옆 칸과 안 섞이게
            tail = _label_lines(val_lines or txt.splitlines(), label)
            cands = _nums("\n".join(tail)) if tail else nums
            fell_back = not tail
        hits = [v for v in cands if lo <= v <= hi]
        stale = _stale_code(code_notes.get(label), src) if hits else ""
        if hits and not stale:
            print(f"✅ {label} — 출력에서 {hits[0]:,.6g} 를 찾았습니다")
            note = _note_for(pass_notes.get(label, ()), [hits[0]], "")
            if note:
                print(f"   ⓘ {note}")
            if unit and unit not in txt:
                print(f"   ⓘ 단위 「{unit}」 도 값과 함께 출력하면 보고서에 그대로 옮길 수 있습니다")
        elif stale:                                   # 값만 맞춰 적고 코드는 그대로 둔 자리
            passed = False
            failed.append((label, label))
            print(f"❌ {label} — 출력에서 {hits[0]:,.6g} 를 찾았지만 {stale}")
        elif table_flood:
            passed = False
            failed.append((label, label))
            print(f"❌ {label} — 「이름 = 숫자 {unit}」 한 행으로 골라 출력하면 채점이 읽습니다")
        else:
            passed = False
            failed.append((label, label))
            where = f"「{label_head}」 이 적힌 행에" if label_head else "출력에"
            _prev_fail = _TRY.get(key, 0)              # 결정 16 — 첫 ❌ 에는 기준 범위를 감춘다
            print(f"❌ {label} — 기준을 충족하는 숫자가 {where} 없습니다"
                  + (f" (기준 범위 {lo:g} ~ {hi:g})" if _prev_fail else ""))
            tails = _label_lines(val_lines or txt.splitlines(), label)
            note = _note_for(fail_notes.get(label, ()), _nums("\n".join(tails)), " ".join(tails))
            if note:                                 # 흔한 실수 하나와 같은 값이면 그 원인을 바로 말합니다
                print(f"   원인 : {note}")
            if fell_back:                            # 라벨 행을 못 찾아 값줄 전체를 본 자리
                print(f"   ⓘ 라벨 「{label}」 이 든 행을 찾지 못해 값줄 전체에서 찾았습니다"
                      " — 「이름 = 숫자 단위」 한 행으로 출력하면 그 행만 보고 채점합니다")
    plot_spec = spec.get("plot")
    if plot_spec:
        axs = [a for f in figs for a in _axes(f)]
        if not axs:
            passed = False
            failed.append(("그래프", "plot"))
            print("❌ 그래프 — 그림을 찾지 못했습니다 (그래프 한 장도 같은 프롬프트에 함께 시킵니다)")
        else:
            need_axes = int(plot_spec.get("n_axes", 1))   # 그림 두 장을 나란히 요구하는 기준이면 2 입니다
            ok_axes, gaps = 0, None
            for ax in axs:                 # 기준에 맞는 축이 필요한 수만큼 있으면 통과입니다
                try:
                    this_gaps = _plot_gaps(ax, plot_spec)
                except Exception as e:
                    this_gaps = [f"그림에서 데이터를 뽑다 걸렸습니다 ({type(e).__name__})"]
                if not this_gaps:
                    ok_axes += 1
                    if ok_axes >= need_axes:
                        break
                elif gaps is None or len(this_gaps) < len(gaps):
                    gaps = this_gaps
            if ok_axes < need_axes:
                passed = False
                failed.append(("그래프", "plot"))
                extra_msgs = [f"기준에 맞는 그림이 {ok_axes}개다 (기준 {need_axes}개)"] if need_axes > 1 else []
                print("❌ 그래프 — " + " · ".join(extra_msgs + (gaps or [])))
            else:
                print("✅ 그래프 — 기준대로 그렸습니다")
            _ref_show(figs[0] if figs else None, key)
    tbl = _HINT.get(key, {})
    attempt = _TRY[key] = _TRY.get(key, 0) + (1 if failed else 0)
    for name, hkey in failed:
        tiers = tbl.get(hkey, ("",))
        tier = tiers[:-1] if len(tiers) > 2 else tiers[:1]   # 2단 힌트는 마지막 원소가 튜터에 물을 것입니다
        hint = tier[min(attempt, len(tier)) - 1] if tier else ""
        if hint:
            print(f"   힌트({name}) : {hint}")
    if failed:
        form = _ASK.get(key, "full")
        print(_ASK_HEAD[form])
        for name, hkey in failed:
            question = (tbl.get(hkey, ("", ""))[-1] if len(tbl.get(hkey, ("",))) > 1 else "") \
                or "이 항목이 왜 ❌ 로 남는지"
            print("  " + _ASK_FORM[form].format(name=name, prompt=_prompt1(key), ask=question))
    elif _locked:                                  # 값·그림은 통과했지만 내 말로 시킨 자리가 아직 비었습니다
        print("ⓘ 값과 그림은 채점 기준을 충족했습니다 — PROMPT 에 고칠 지시를 내 말로 쓰고"
              " 다시 실행하면 완성본 풀이가 열립니다")
    else:
        print("🔓 정답이 열렸습니다 — 아래 「🔓 완성본 풀이 보기」를 펼쳐 내 코드와 견줘 보세요")
        _reveal_show(base64.b64decode(_REVEAL[key]).decode("utf-8"))
    print("─" * 54)
    return passed and not _locked


class _Tee:
    """「PROMPT 셀」의 출력을 화면과 버퍼 양쪽으로 흘려보냅니다"""

    def __init__(self, orig, buf):
        self._o, self._b = orig, buf

    def write(self, s):
        self._b.write(s)
        return self._o.write(s)

    def flush(self):
        self._o.flush()

    def __getattr__(self, n):
        return getattr(self._o, n)


def _tag(src):
    """첫 줄이 `# 미션 N — 프롬프트` · `# 과제 — 프롬프트` 인 셀만 채점 대상입니다"""
    head_line = (src or "").split("\n")[0]
    m = re.match(r"\s*#\s*(미션\s*(\d+)|과제)\b", head_line)
    if not m or "프롬프트" not in head_line:                     # 「준비 셀」(`# 미션 N · 준비`)은 대상이 아닙니다
        return None
    return "m" + m.group(2) if m.group(2) else "과제"


def _pre(info=None):
    """「PROMPT 셀」면 출력 버퍼를 걸고 그림 목록을 비웁니다"""
    try:
        tag = _tag(getattr(info, "raw_cell", ""))
        _STATE["tag"] = tag
        if tag is None or tag not in _ANS:
            return
        _FIGS.clear()
        _STATE["ids0"] = {id(f) for f in _open_figs()}
        _STATE["buf"], _STATE["out"] = io.StringIO(), sys.stdout
        sys.stdout = _Tee(sys.stdout, _STATE["buf"])
    except Exception:
        pass


def _post(result=None):
    """「PROMPT 셀」이 끝나면 출력을 되돌리고 그 셀의 출력·그림으로 채점합니다"""
    tag, buf, out = _STATE.get("tag"), _STATE.get("buf"), _STATE.get("out")
    _STATE["tag"], _STATE["buf"], _STATE["out"] = None, None, None
    if out is not None:
        sys.stdout = out
    if tag is None or tag not in _ANS:
        return
    seen = {id(f) for f in _FIGS}
    figs_all = list(_FIGS) + [f for f in _open_figs()
                         if id(f) not in seen and id(f) not in _STATE.get("ids0", set())]
    txt = buf.getvalue() if buf is not None else ""
    last_val = getattr(result, "result", None)       # 셀 마지막 표현식의 값 — print 없이 표로 띄운 값도 읽습니다
    if last_val is not None:
        try:
            txt = txt + "\n" + str(last_val)
        except Exception:
            pass
    try:
        judge(tag, txt, figs_all, error=getattr(result, "error_in_exec", None),
              src=getattr(getattr(result, "info", None), "raw_cell", "") or "")
    except Exception as e:
        print(f"(채점을 마치지 못했습니다 — {type(e).__name__}: {_one_line(e)})")


try:
    _ip = get_ipython()
    for _ev, _fn in (("pre_run_cell", _pre), ("post_run_cell", _post)):
        for _old in list(_ip.events.callbacks[_ev]):
            if getattr(_old, "__name__", "") == _fn.__name__:
                _ip.events.unregister(_ev, _old)     # 이 셀을 다시 실행해도 채점은 한 번만 찍힙니다
        _ip.events.register(_ev, _fn)
    print("채점 준비 완료 — 미션의 「PROMPT 셀」을 실행하면 실행 결과(출력)에 채점 결과가 찍힙니다")
except Exception:
    print("채점 준비 완료 (IPython 밖이라 훅 없이 judge() 만 올렸습니다)")

## 1. 168시간 예측은 계절 나이브 168 보다 오차가 작은가

### 미션 1 [필수] — 168시간을 채우고 계절 나이브 168 보다 오차가 작은지 확인하기 (함정 미션)

#### 할 일 — 미션 1 (40분)

1. 아래 「준비 셀」을 실행합니다. 이 셀은 고치지 않습니다.
2. 「PROMPT 셀」을 먼저 그대로 ▶ 해 전임 담당자 코드의 채점 결과(❌)를 봅니다. 고치는 것은 3번부터입니다. PROMPT 가 비어 있어도 이 채점은 출력됩니다.
3. 아래 고칠 것대로 PROMPT 를 쓰고, 셀을 클릭한 채 AI 튜터에게 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶.
4. 실행 결과에 출력되는 채점 결과를 봅니다. ❌ 면 힌트대로 PROMPT 를 고쳐 3번부터 다시 합니다.

**이 미션의 질문**: SARIMA 모형이 만든 168시간 표는 계절 나이브 168 보다 오차가 작습니까?

**쓸 수 있는 것**: 「준비 셀」이 만들어 둔 이름입니다.

| 이름 | 무엇 | 다룸 |
|---|---|---|
| `y_train` | 학습 구간(11월 16일 23시까지) 보정 수요 1,848시간, 단위 대/시간 | 고정 |
| `y_valid` | 검증 구간(17일 0시부터) 실측 168시간, 단위 대/시간 | 고정 |
| `df` | 시간당 기록 표 전체, 보정 수요는 `demand` 열 | 고정 |
| `ORDER` · `SEASONAL_ORDER` | SARIMA(1, 0, 1)(1, 1, 1)₂₄ 의 앞 괄호와 뒤 괄호 | 그대로 전달한다 |
| `FIT_OPTS` | 적합 옵션 문자열 | 그대로 적는다 |
| `SARIMAX` · `rmse` · `datefmt` | 적합 함수 · 오차 함수 · 가로축 날짜 표기 함수 | 그대로 호출한다 |

#### 고칠 것

먼저 전임 코드를 그대로 실행해 출력된 값 2행을 아래 빈칸에 옮겨 적고, 고친 뒤 그 2행이 어떻게 달라지는지를 답으로 삼습니다. RMSE 는 시각마다 빗나간 크기를 한 수로 모은 오차입니다(아래 「결과를 읽을 때 쓰는 개념」).

> **전임 코드가 출력한 두 RMSE:** ____

고친 뒤 아래가 모두 출력되게 합니다. 굵은 이름은 글자 그대로 사용합니다.

1. **검증 168시간 RMSE**: 단위 대 · 소수점 아래 2자리 · SARIMA 모형이 168시간을 한 번에 예측해 검증 구간에서 받은 값
   - 전임 코드가 이미 맞게 구한 값입니다
2. **1주일 전 값 규칙의 RMSE**: 단위 대 · 소수점 아래 2자리 · `demand` 를 168시간 뒤로 이동시켜 만든 예측이 같은 168시간에서 받은 값
   - 고칠 곳 : 전임 코드가 `demand` 를 뒤로 이동시킨 시간 수 하나입니다. 24시간 뒤로 옮기면 18일 8시(33시간 뒤) 예측에 17일 8시 실측이 들어가는데, 예측을 시작하는 16일 23시에는 없는 값입니다. 168시간이면 17일 8시 예측에 10일 8시 실측이 들어가 168개 모두 16일 23시 이전 값입니다
   - 같은 규칙을 본문에서 부르는 이름은 계절 나이브 168 입니다
3. **그래프 한 장**: 가로축은 검증 구간 시각, 세로축은 수요(대)인 선 그래프
   - 선 : 2개(검증 168시간의 실측선 하나와 같은 구간의 예측선 하나). 두 선은 다른 색으로 그리고, 범례로 어느 쪽이 실측인지 표시합니다
   - 채움 영역 : 95% 예측구간의 아래쪽 경계와 위쪽 경계 사이를 옅은 색으로 채웁니다
   - 가로축 이름 : 「검증 구간 시각」 · 날짜 표기는 `datefmt(ax, "%m-%d", 8)` 처럼 그래프를 그린 `ax` 와 표기와 눈금 수를 전달해 정리합니다
   - 세로축 이름 : 「수요(대)」
   - 전임 코드가 이미 이 기준대로 그렸습니다

**출력 형식과 통과 기준**: 값 2개를 「이름 = 값 대」 형태로 한 행씩 `print` 하고, 그래프는 한 장입니다. 굵은 이름 2개가 각각 자기 행에 있고 값이 기준 범위에 들며 선 그래프가 위 요구를 갖추면 통과입니다.

**채점이 보지 않는 곳**: 그림의 예측구간은 `conf_int(alpha=0.05)` 로 받은 95% 구간이어야 합니다. 다른 값으로 받아도 오류 없이 통과하므로 직접 확인합니다.

#### 결과를 읽을 때 쓰는 개념

- **RMSE**: 시각마다 (실측 − 예측)을 제곱해 168개를 평균 낸 뒤 제곱근을 씌운 값입니다. 작을수록 좋고, 200대라면 한 시간에 대략 200대씩 빗나간다는 뜻입니다.
- **계절 나이브 168**: 1주일 전 같은 시각 실측을 그대로 예측으로 내는 기준선입니다. RMSE 하나로는 잘했는지 알 수 없어 이 기준선과 비교합니다. 17일(토) 0시 예측은 10일 0시 실측 754대이고, 실측은 626대였습니다.
- **같은 조건 비교**: 두 예측 모두 원점(예측을 시작한 시각) 11월 16일 23시까지의 값만 씁니다. 같은 검증 168시간에서 채점해야 두 RMSE 를 비교할 수 있습니다.

> **고친 뒤의 두 RMSE:** ____

> **이번 주 배차표의 숫자를 가져올 쪽:** ____ (근거가 된 두 RMSE 와, 검증한 주가 11월 17～23일 한 주뿐이라는 한계를 한 문장으로)

#### 프롬프트에 넣을 것(힌트)

- **고칠 곳**: `demand` 를 몇 시간 뒤로 이동시킬지 PROMPT 에 숫자로 적습니다. 그 숫자 하나가 이 미션의 답입니다.
- **손대지 않을 범위**: 적합과 168시간 예측과 95% 예측구간을 받는 부분, 그리고 이미 그려진 선 그래프는 그대로 두라고 적습니다.
- **고정할 이름**: `y_train` · `y_valid` · `ORDER` · `SEASONAL_ORDER` 는 이미 있으므로 다시 만들지 말라고 적습니다.
- **출력**: 값 2개의 이름과 단위, 소수점 아래 몇 자리까지 출력할지.

**막히면**: AI 튜터에게 「1주일 전 같은 시각 값을 예측으로 쓰려면 시간 계열을 몇 시간 뒤로 이동시켜야 하나요」 처럼 질문하세요.

#### 예상 적기

하나를 고르고, 그렇게 판단한 이유를 한 문장 적으세요.

> **[예측] 고친 뒤 같은 168시간에서 오차가 더 작은 쪽은 어디일까요?**
>
> 1. SARIMA 모형: 계수 4개를 1,848시간으로 추정해 하루 모양과 수준을 함께 따라간다
> 2. 계절 나이브 168: 1주일 전 같은 시각의 실측을 그대로 써서 최근 수준을 따라간다
> 3. 거의 같다: 두 예측 모두 8시와 18시에 많은 하루 모양을 따라가므로 오차 차이가 몇 대에 그친다
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

<details class="lms-extra"><summary>자세히 — 168시간을 한 번에 예측하는 뜻, 예측구간, 계절 나이브 그림</summary>

신뢰구간은 평균 수준의 범위라, 관측 하나가 들어갈 **예측구간**과 답하는 질문이 다릅니다.

168시간을 한 번에 예측한다는 말은 중간에 실측을 다시 입력하지 않는다는 뜻입니다. 11월 23일 23시를 예측할 때도 모형이 사용한 마지막 실측은 11월 16일 23시 하나입니다.

<img src="https://otexts.com/fpp3/fpp_files/figure-html/traintest-1.png" width="640" height="64"/>
<p align="center"><em>▲ 계열 하나를 달력 순서 그대로 가른 그림. 앞쪽 파랑이 학습, 뒤쪽 주황이 채점이다. 오늘은 2018년 9월 1일 0시부터 11월 16일 23시까지 1,848시간이 파랑이고 그 뒤 168시간이 주황이다 (출처: fpp3 §5.8)</em></p>

<img src="https://otexts.com/fpp3/fpp_files/figure-html/snaive-method-explained-1.png" width="640" height="352"/>
<p align="center"><em>▲ 계절 나이브. 마지막 한 주기의 값 4개인 진한 파랑 점을 그대로 이어 연한 파랑 예측을 만든다. 오늘 쓰는 계절 나이브 168 은 그 주기가 168시간이라 1주일 전 같은 시각 값이 그대로 예측이 된다 (출처: fpp3 §5.2)</em></p>

</details>

<img src="https://otexts.com/fpp3/fpp_files/figure-html/usemployment7-1.png" width="640" height="320"/>
<p align="center"><em>▲ 계절 패턴을 이어 그리는 SARIMA 예측. 파란 선이 점 예측이고, 점 예측을 가운데 두고 진한 영역이 80% · 옅은 영역이 95% 예측구간이다. 먼 시각을 예측할수록 예측구간이 길어진다 (출처: Hyndman &amp; Athanasopoulos, fpp3 §9.9)</em></p>

In [ ]:
# 미션 1 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 1 에 쓸 변수를 직접 만듭니다
if "df" not in globals() or "demand" not in df.columns:   # 이 미션만 따로 열어도 단독으로 실행됩니다
    URL = "https://archive.ics.uci.edu/static/public/560/seoul+bike+sharing+demand.zip"
    raw = pd.read_csv(URL, encoding="cp1252", compression="zip")      # 8,760행 x 14열
    #  받아지지 않으면: UCI 페이지에서 zip 을 내려받아 Colab 왼쪽 파일 영역(폴더 아이콘)에 올린 뒤
    #  URL 대신 "seoul+bike+sharing+demand.zip" 을 넣으세요
    raw.columns = ["date", "count", "hour", "temp", "humidity", "wind", "visibility",   # 위치 기반 리네임
                   "dewpoint", "solar", "rain", "snow", "season", "holiday", "functioning"]
    raw["datetime"] = (pd.to_datetime(raw["date"], format="%d/%m/%Y")   # 01/12/2017 = 2017년 12월 1일
                       + pd.to_timedelta(raw["hour"], unit="h"))
    df = raw.sort_values("datetime").set_index("datetime")
    df.index.freq = "h"                   # 등간격 1시간 — 168시간 뒤로 이동시키는 것이 "1주일 전" 이라는 뜻이 된다
    _s = df["count"].astype(float).mask(df["functioning"].eq("No"))   # 휴무는 0 이 아니라 관측 없음
    _y = _s.fillna(_s.shift(24 * 7))      # 1주일 전 같은 시각 값으로 채운다 (요일 효과를 지키려고)
    df["demand"] = _y.fillna(_y.shift(-24 * 7))   # 1주일 전도 휴무인 시각만 1주일 뒤 값으로
    df["is_imputed"] = df["functioning"].eq("No")

FIT_START, TRAIN_END, VAL_END = "2018-09-01 00:00", "2018-11-16 23:00", "2018-11-23 23:00"
VAL_START = pd.Timestamp(TRAIN_END) + pd.Timedelta(hours=1)   # 검증은 학습이 끝난 바로 다음 시각부터
y_train = df["demand"].loc[FIT_START:TRAIN_END]     # 쓸 변수 1 — 학습 구간 보정 수요 (대/시간)
y_valid = df["demand"].loc[VAL_START:VAL_END]       # 쓸 변수 2 — 검증 구간 실측 (대/시간)

ORDER, SEASONAL_ORDER = (1, 0, 1), (1, 1, 1, 24)    # 오늘 쓸 차수: SARIMA(1, 0, 1)(1, 1, 1)₂₄
LAST_SEASONAL = (0, 1, 0, 24)                       # 어제 차수의 뒤 괄호: 계절차분만 있고 계절 AR·MA 가 없다
FIT_OPTS = "trend=None, enforce_stationarity=True, enforce_invertibility=True"

_fits = {}                                          # 두 차수를 같은 학습 구간에 각각 한 번씩 적합한다
for _name, _seas in (("어제", LAST_SEASONAL), ("오늘", SEASONAL_ORDER)):
    _fits[_name] = SARIMAX(y_train, order=ORDER, seasonal_order=_seas, trend=None,
                           enforce_stationarity=True, enforce_invertibility=True).fit(disp=False)
_aic = {_name: _fit.aic for _name, _fit in _fits.items()}
_last_pred = _fits["어제"].get_forecast(steps=len(y_valid)).predicted_mean   # 어제 차수로 같은 168시간을 한 번에

print(f"학습 {FIT_START} ~ {TRAIN_END}   {len(y_train):,}시간   평균 {y_train.mean():,.1f} 대/시간   "
      f"채운 시각 {int(df['is_imputed'].loc[FIT_START:TRAIN_END].sum())}시간")
print(f"검증 {VAL_START:%Y-%m-%d %H:%M} ~ {VAL_END}   {len(y_valid):,}시간   "
      f"평균 {y_valid.mean():,.1f} 대/시간   채운 시각 {int(df['is_imputed'].loc[VAL_START:VAL_END].sum())}시간")
print(f"오늘 쓸 차수 : 앞 괄호 ORDER = {ORDER} · 뒤 괄호 SEASONAL_ORDER = {SEASONAL_ORDER}")
print(f"어제 차수 {ORDER}{LAST_SEASONAL} AIC = {_aic['어제']:,.2f}"
      f"   →   오늘 차수 {ORDER}{SEASONAL_ORDER} AIC = {_aic['오늘']:,.2f}")
print(f"계절 항 2개를 더해 AIC 가 {_aic['어제'] - _aic['오늘']:,.2f} 내려갔습니다 —"
      f" 두 적합 모두 같은 학습 {len(y_train):,}시간입니다")
print(f"어제 차수로 같은 검증 168시간을 한 번에 예측한 RMSE = {rmse(y_valid, _last_pred):,.2f} 대"
      " — 고친 뒤 오늘 차수의 값과 비교합니다")
print(f"적합 옵션 통일 — {FIT_OPTS}")
print("보정 수요 df['demand'] 는 전 구간이 결측 없이 채워져 있습니다 —"
      " 비교할 규칙이 검증 시각 이전 값을 참조해야 하기 때문입니다.")
_p, _e = _fits["오늘"].params, _fits["오늘"].resid       # 오늘 모형의 계수 4개와 한 시간 뒤 예측 오차(잔차)
_d = df["demand"]; _z = _d - _d.shift(24)                # z = 지금 값 − 24시간 전 같은 시각 값
_h = pd.Timedelta(hours=1); _t = VAL_START                # 11월 17일 0시 예측을 항별로 나눠 본다
print(f"오늘 모형의 계수 4개 : φ₁ {_p['ar.L1']:.2f} · θ₁ {_p['ma.L1']:.2f} · Φ₁ {_p['ar.S.L24']:.2f} · Θ₁ {_p['ma.S.L24']:.2f}")
_terms = {"24시간 전 실측": _d[_t - 24*_h], "φ₁×z(1시간 전)": _p["ar.L1"] * _z[_t - _h],
          "θ₁×오차(1시간 전)": _p["ma.L1"] * _e[_t - _h], "Φ₁×z(24시간 전)": _p["ar.S.L24"] * _z[_t - 24*_h],
          "Θ₁×오차(24시간 전)": _p["ma.S.L24"] * _e[_t - 24*_h],
          "곱 항 2개": -_p["ar.L1"] * _p["ar.S.L24"] * _z[_t - 25*_h] + _p["ma.L1"] * _p["ma.S.L24"] * _e[_t - 25*_h]}
print("11월 17일 0시 예측을 항별로 :", " · ".join(f"{k} {v:+,.1f}" for k, v in _terms.items()),
      f"= {sum(_terms.values()):,.1f} 대 (모형 출력 {_fits['오늘'].get_forecast(1).predicted_mean.iloc[0]:,.1f} 대)")
print("   손으로 더한 값과 모형 출력이 1대쯤 다른 것은 모형이 학습 첫 시각부터 이어 온 계산 상태로 예측하기 때문입니다.")
print("이 셀이 두 번 적합하므로 10초 남짓 걸립니다. 그대로 두고 기다리세요.")
print("쓸 수 있는 이름 : df (표 전체, 보정 수요는 demand 열) · y_train (학습 구간 보정 수요, 대/시간)"
      " · y_valid (검증 구간 실측, 대/시간) · ORDER · SEASONAL_ORDER (오늘 쓸 차수 2벌)"
      " · FIT_OPTS (적합 옵션 문자열) · SARIMAX · rmse(실측, 예측) · datefmt(ax, 표기, 눈금 수)")

In [ ]:
# 미션 1 — 프롬프트 (함정)   # ai: prompt
# 이 미션은 함정입니다 — 아래는 전임 담당자가 남긴 코드라 돌아가지만 값 하나가 틀렸습니다.
# 어디를 어떻게 고칠지 PROMPT 에 쓰면 튜터가 AI 코드 표시선 아래를 다시 씁니다.
PROMPT = """

"""
# --- 여기부터 AI 코드 ---
# ↓ 전임 담당자의 코드 — 그대로 실행하면 채점이 ❌ 를 냅니다
res = SARIMAX(y_train, order=ORDER, seasonal_order=SEASONAL_ORDER, trend=None,
              enforce_stationarity=True, enforce_invertibility=True).fit(disp=False)
fc = res.get_forecast(steps=len(y_valid))       # 학습 구간 끝 다음 시각부터 한 번에 168시간
ci = fc.conf_int(alpha=0.05)                    # 95% 예측구간
pred = np.asarray(fc.predicted_mean, dtype=float)

naive = df["demand"].shift(24).loc[y_valid.index]   # 1주일 전 같은 시각 값을 그대로 산출하는 규칙
print(f"검증 168시간 RMSE = {rmse(y_valid, pred):,.2f} 대")
print(f"1주일 전 값 규칙의 RMSE = {rmse(y_valid, naive):,.2f} 대")

fig, ax = plt.subplots(figsize=(FIG_W, 3.8))
ax.fill_between(y_valid.index, ci.iloc[:, 0], ci.iloc[:, 1], color=COL["하늘색"], alpha=0.18)
ax.plot(y_valid.index, y_valid, color=COL["남색"], lw=1.4, label="실측")
ax.plot(y_valid.index, pred, color=COL["하늘색"], lw=1.8, label="예측과 95% 예측구간")
ax.legend()
ax.set_xlabel("검증 구간 시각")
ax.set_ylabel("수요(대)")
datefmt(ax, "%m-%d", 8)
plt.show()

> **[문제] 계절 항 2개를 더해 AIC 가 706.65 내려간 것을 읽기**
>
> 「준비 셀」은 어제 차수 (1, 0, 1)(0, 1, 0, 24) 의 AIC 25,209.34 와 오늘 차수 (1, 0, 1)(1, 1, 1, 24) 의 AIC 24,502.69 를 함께 출력했습니다. 두 값의 차이 706.65 를 읽는 방법으로 맞는 것은?
>
> 1. 706.65 는 검증 168시간 RMSE 가 줄어든 양이라, 오늘 예측의 오차도 어제보다 그만큼 작아진다
> 2. 같은 1,848시간에서 계산한 점수라 비교할 수 있고, 차이가 2 를 크게 넘어 계절 항을 더한 쪽을 고른다
> 3. AIC 는 값이 클수록 좋은 점수라, 25,209.34 를 받은 어제 차수가 계절 항 없이도 더 나은 모형이다

> **[문제] 미션 1 의 출력에는 SARIMA 모형의 RMSE 와 계절 나이브 168 의 RMSE 가 나란히 출력되었습니다. 두 숫자를 비교하는 방법으로 맞는 것은?**
>
> 1. 규칙 쪽은 검증 구간 안의 실측을 그대로 가져다 쓰는 예측이라, SARIMA 모형과 나란히 비교할 수 없다
> 2. 둘 다 11월 16일 23시에서 168시간을 예측해 비교할 수 있고, SARIMA 모형의 오차가 더 컸다
> 3. 규칙 쪽은 1시간 뒤 예측이라 SARIMA 모형만 불리했을 뿐, 같은 시간 앞으로 맞추면 순위가 곧바로 반대가 된다

> **[문제] 미션 1 의 그림에서 예측선 위아래를 채운 옅은 영역이 95% 예측구간입니다. 11월 17일 0시의 영역은 241대부터 1,008대까지인데, 이 범위를 읽는 방법으로 맞는 것은?**
>
> 1. 11월 17일 0시 수요의 평균이 241대와 1,008대 사이에 있을 확률이 95% 라는 뜻이다
> 2. 검증 168시간의 실측 96.4% 가 들어오도록 실측을 보고 길이를 정한 범위다
> 3. 11월 17일 0시에 기록될 수요 한 값이 그 사이에 들어올 확률을 95% 로 본 범위다

## 2. 인수인계표 — 168시간의 확신과 보류 구분

> **[과제] 과제 제출 — 168시간 중 어디까지를 확신 등급으로 전달할 것인가**
>
> - **무엇을** 배차팀에 전달할 168시간 인수인계표와 그 표를 그린 그림 한 장을 만듭니다.
> - **무엇으로** 오늘 사용한 데이터 하나만 사용하고, 「준비 셀」이 2018-11-17 00시부터 168시간 예측표와 잔차까지 만들어 둡니다.
> - **인정** 채점이 값 6개와 그림을 모두 통과시킨 노트북에 브리핑 세 문장을 채운 것입니다.
> 
> - 인수인계표 한 행은 「11월 17일 08시 · 예측 ○○○ 대 · 하한 ○○○ · 상한 ○○○ · 구간 길이 ○○○ · 확신」 형태입니다.
> - 확신과 보류는 「확신 = ○○ 시간 · 보류 = ○○ 시간」처럼 값과 같은 출력 행에 적습니다.
> - 「PROMPT 셀」을 실행해 값 6개와 그림 한 장이 모두 채점을 통과한 상태로 남깁니다.
> - 확신과 보류를 구간 길이로 구분하고, 그 경계가 되는 구분 기준값을 내가 정해 PROMPT 에 숫자로 적으며, 그 값을 정한 운영 근거도 한 구로 밝힙니다.
> - 하한이 음수로 나온 시각을 어떻게 보고할지 정해 브리핑에 밝힙니다.
> - 브리핑 세 문장을 과제 마지막의 「브리핑 세 문장」 양식에 결론 · 위험 · 코드가 답을 만드는 방식 순으로 채웁니다.
>
> 마감: 2026-10-08T23:59+09:00
>
> **평가 기준**
>
> | 기준 | 이렇게 하면 충족 | 배점 |
> | --- | --- | --- |
> | 문제 정의 | PROMPT 에 목표와 쓸 변수와 출력 형식과 내가 고른 구분 기준값이 숫자로 적혀 있고, 그 값을 정한 운영 근거가 한 구로 함께 적혀 있다. | 25 |
> | 검증 설계 | 값 6개와 그림이 채점을 통과하고 확신과 보류의 합이 168시간이다. | 25 |
> | 오류 탐지 | 하한이 음수인 시각을 집계하고 그 원인을 모형의 분포 가정으로 설명했으며, 그 시각을 표에 어떻게 적어 전달했는지 함께 밝혔다. | 25 |
> | 해석·보고 | 잔차 검정 결과를 한계로 적고 경계가 배차에 무엇을 뜻하는지 밝혔다. | 25 |
>
> 제출은 LMS 레슨 화면의 과제 카드에서 GitHub 또는 Google Drive 링크로 합니다(이 파일에 적으면 제출되지 않습니다).

### 과제 — 168시간 중 어디까지를 확신 등급으로 전달할 것인가

#### 할 일 — 과제 (40분)

1. 「준비 셀」 실행부터 채점 결과 확인까지 미션 1 과 같은 순서이고, 구분선 아래에 전임 코드가 없어 PROMPT 부터 씁니다.
2. 완성본의 숫자는 구분 기준값 예시 하나에서 나온 값이라 내 숫자와 달라도 됩니다. 완성본과 대조할 때에는 세 문장이 결론·위험·방식을 각각 적었는지만 봅니다.

**이 과제의 질문**: 배차팀에 전달할 168시간 가운데 어디까지를 확신으로 적을 수 있습니까?

**쓸 수 있는 것**: 「준비 셀」이 만들어 둔 이름입니다.

| 이름 | 무엇 | 다룸 |
|---|---|---|
| `fc` | 168행 예측표: `actual` · `pred` · `lo95` · `hi95` · `width` 열 5개이고 `width` 가 구간 길이입니다 | 고정 |
| `resid` | 그 모형의 잔차 | 고정 |
| `y_valid` | 검증 구간 실측 168시간(보정 수요 `demand` 열), 단위 대/시간 | 고정 |
| `df` | 시간당 기록 표 전체 | 고정 |
| `acorr_ljungbox` | 잔차에 자기상관이 남았는지 시차 여럿을 한 번에 묻는 검정 함수 | 그대로 호출한다 |
| `datefmt` | 가로축 날짜 표기 함수 | 그대로 호출한다 |

#### 만들 것

「PROMPT 셀」 한 곳에서 아래 1번 표, 2～7번 값 6개, 8번 그림이 모두 출력되게 합니다. 굵은 이름은 글자 그대로 씁니다.

1. **인수인계표**: 검증 168시간마다 시각 · 예측 · 하한 · 상한 · 구간 길이 · 등급까지 열 6개를 갖춰 출력합니다. 한 행은 「11월 17일 08시 · 예측 ○○○ 대 · 하한 ○○○ · 상한 ○○○ · 구간 길이 ○○○ · 확신」 형태입니다
   - 표의 예측·하한·상한은 SARIMA 값입니다(17일 8시 예측 1,446대, 배차에 쓸 계절 나이브 168 은 589대, 실측 530대). 계절 나이브 168 에는 오늘 구간이 없어, 등급은 먼 시각일수록 길어지는 SARIMA 구간으로 정합니다. 등급은 589대의 빗나감을 직접 계산한 값이 아닙니다. 확신은 그 대수를 금요일에 확정할 시각이고, 보류는 당일 아침 실측을 보고 다시 정할 시각입니다. 구간 길이가 **구분 기준값**보다 짧은 시각이 확신, 나머지가 보류입니다
   - 구분 기준값과 그 운영 근거 한 구는 내가 정합니다. 구간 길이 1,200대는 실측이 예측의 위아래 600대 안에 95% 확률로 든다는 뜻입니다. 「준비 셀」이 출력한 구간 길이 사이에서, 당일 아침 재배치가 감당할 수 있는 빗나감으로 고릅니다. 구분 기준값이 1,200대면 2시간, 1,500대면 6시간, 1,550대면 15시간, 1,600대면 26시간이 확신입니다. 가장 짧은 17일 0시 구간 767대도 예측 625대보다 깁니다. 확신은 그 수를 그대로 믿어도 된다는 뜻이 아니라, 이 표 안에서 구간이 짧은 앞쪽 시각이라는 뜻입니다. 한쪽이 0행이면 통과하지 않습니다
2. **인수인계표 행 수**: 단위 행 · 정수
3. **확신으로 남은 시각 수**: 단위 시간 · 정수 · 「확신 = ○○ 시간」처럼 그 단어를 값과 같은 출력 행에 둡니다
4. **보류로 남은 시각 수**: 단위 시간 · 정수 · 「보류 = ○○ 시간」처럼 그 단어를 값과 같은 출력 행에 둡니다
5. **하한이 음수인 시각 수**: 단위 시간 · 정수 · `lo95` 가 0 보다 작은 행의 개수
6. **잔차 Ljung-Box 시차 24 통계량**: 소수점 아래 2자리 · `resid` 에 시차 24 · `model_df` 4 로 적용한 검정의 `lb_stat`
7. **잔차 Ljung-Box 시차 24 p 값**: `3.2e-47`(3.2 × 10⁻⁴⁷) 같은 지수 표기 · 같은 검정의 `lb_pvalue`
8. **그래프 한 장**: 가로축은 검증 구간 시각, 세로축은 수요(대)인 선 그래프
   - 선 : 168시간의 예측선 1개이고, 실측선을 겹쳐 그려도 됩니다
   - 채움 영역 : 95% 예측구간의 아래쪽 경계와 위쪽 경계 사이를 옅은 색으로 채웁니다
   - 세로선 : 등급이 처음 보류로 바뀌는 시각에 세로축 전체에 걸쳐 1개
   - 가로축 이름 : 「검증 구간 시각」 · 세로축 이름 : 「수요(대)」

#### 조건

`model_df` 는 모형이 추정한 계수 수라 4 입니다. 1시간 전과 24시간 전 같은 시각의 $z$·잔차에 곱하는 계수 4개입니다. 검정의 자유도(자기상관이 없을 때 통계량의 평균)는 그만큼 빼서 24 − 4 = 20 입니다.

- **출력 형식**: 값 6개를 「이름 = 값 단위」 형태로 `print` 합니다. 확신과 보류는 한 행에 나란히 두어도, 두 행으로 나눠도 됩니다. 그래프는 한 장이고, 인수인계표는 화면에만 출력하며 파일로 내보내지 않습니다.
- **통과 기준**: 값 6개가 출력되고 그림이 위 요구를 갖추면 통과입니다. 인수인계표 자체는 채점하지 않고 2번의 행 수로 확인합니다.
- **확인할 것**: 3번과 4번의 합이 168 이 아니면 등급 열이 일부 행을 비워 둔 것입니다.

#### 결과를 읽을 때 쓰는 개념

- **95% 예측구간**: 잔차가 정규분포를 따른다고 보고 예측값에 같은 길이를 빼고 더한 범위입니다. 먼 시각일수록 길어지고, 예측값이 구간 길이의 절반보다 작은 시각(0～6시와 10시)에는 하한이 0 밑으로 내려갑니다. 17일 4시는 예측 131대에 733대를 빼고 더해 −602～863대였습니다.
- **Ljung-Box 검정**: 잔차에 시차 1 부터 지정한 시차까지 자기상관이 남았는지 한 번에 묻습니다. p 값이 작으면 모형이 설명하지 못한 규칙이 잔차에 남았다는 결론입니다.

#### 프롬프트에 넣을 것(힌트)

- **구분 기준값**: 구간 길이를 어느 수와 비교할지 PROMPT 안에 숫자로 명시합니다. 그 문장이 없으면 표는 나오는데 등급 열이 비고, 세로선을 어디에 그을지도 정해지지 않습니다.
- **고정할 이름**: `fc` · `resid` · `y_valid` 는 이미 있으므로 다시 적합하지 말라고 적습니다.

**막히면**: AI 튜터에게 「등급 열을 어떤 열의 어떤 비교로 만들면 확신과 보류가 양쪽 모두 남나요」 처럼 질문하세요.

<details class="lms-extra"><summary>자세히 — AI 튜터에게 맡길 일과 내가 확인할 일</summary>

아래 표의 「내가 반드시 확인할 일」 3가지는 **틀려도 오류가 나지 않고 채점도 통과합니다.**

| AI 튜터에게 맡길 일 | 내가 반드시 확인할 일 |
|---|---|
| 등급 열과 인수인계표 만들기 | 등급을 `width` 와 내가 정한 구분 기준값의 비교로 갈랐는가 |
| Ljung-Box 검정 호출 | 시차 24 와 `model_df` 4 를 그대로 전달했는가 |
| 그림 그리기 | 세로선을 등급이 처음 보류로 바뀌는 시각에 그었는가 |

</details>

<img src="https://otexts.com/fpp3/fpp_files/figure-html/egyptexportsf-1.png" width="640" height="320"/>
<p align="center"><em>▲ 먼 시각을 예측할수록 점 예측은 완만해지고 예측구간만 길어지는 모습(가로축 연도, 파란 선이 예측, 진한 영역이 80% · 옅은 영역이 95% 예측구간) (출처: Hyndman &amp; Athanasopoulos, fpp3 §9.8)</em></p>

<details class="lms-extra"><summary>자세히 — 구간 길이와 상대 구간 길이가 만드는 서로 다른 경계</summary>

구간 길이 자체로 구분하면 **경계가 몇 시간 뒤인지를 따라 생깁니다**. 구간 길이는 시간이 갈수록 커지므로 앞쪽 시각이 확신이 되고, 그림에는 세로선 하나가 남습니다. 오늘 과제가 구간 길이로 등급을 가르는 이유가 이것입니다.

구간 길이를 예측값으로 나눈 **상대 구간 길이**로 구분하면 **경계가 시각을 따라 생깁니다**. 예측값이 작은 새벽이 보류로, 이용자가 많은 낮 시간대가 확신으로 남고, 등급이 하루에도 여러 번 바뀌어 경계를 세로선 하나로 그을 수 없습니다. 17일 4시는 예측 131대에 구간 길이 1,465대라 상대 구간 길이가 11.2 이고, 8시는 1,446대에 1,539대라 1.06 입니다. 어느 쪽이 나은지는 배차 운영에 따라 다릅니다.

</details>

<details class="lms-extra"><summary>자세히 — 하한이 음수인 시각을 적는 두 방법과 잔차 진단 그림</summary>

이론이 적은 조치는 2가지입니다. 하나는 수요에 로그를 씌워 구간을 만든 뒤 복원하는 것으로, 복원한 하한은 0 보다 작아지지 않습니다. 다른 하나는 하한을 0 으로 적고 0 으로 잘랐다는 사실을 표에 함께 남기는 것입니다.

오늘은 로그 변환을 하지 않았으니, 그 시각을 어떻게 적어 전달할지는 내가 정합니다.

<img src="https://otexts.com/fpp3/fpp_files/figure-html/tsresiduals-1.png" width="640" height="395"/>
<p align="center"><em>▲ 어느 예측법의 잔차 진단 그림. 위는 잔차를 시간 순으로, 아래 왼쪽은 잔차 자기상관, 오른쪽은 잔차의 히스토그램이다. 오른쪽 꼬리가 길어 정규분포를 전제로 계산한 예측구간의 길이를 그대로 신뢰하기 어렵다 (출처: Hyndman &amp; Athanasopoulos, fpp3 §5.4)</em></p>

</details>

<img src="https://commons.wikimedia.org/wiki/Special:FilePath/Bicycle-sharing%20station%20in%20Seoul%20Sinseol-dong.jpg?width=1600" width="640" height="539"/>
<p align="center"><em>▲ 지하철 신설동역 출입구 옆 공유자전거 정류소 (출처: Wikimedia Commons, Bicycle-sharing station in Seoul Sinseol-dong)</em></p>

In [ ]:
# 과제 · 준비 — 읽고 실행만 합니다. 이 셀이 과제에 쓸 변수를 직접 만듭니다
if "df" not in globals() or "demand" not in df.columns:   # 과제만 따로 열어도 단독으로 실행됩니다
    URL = "https://archive.ics.uci.edu/static/public/560/seoul+bike+sharing+demand.zip"
    raw = pd.read_csv(URL, encoding="cp1252", compression="zip")      # 8,760행 x 14열
    #  받아지지 않으면: UCI 페이지에서 zip 을 내려받아 Colab 왼쪽 파일 영역(폴더 아이콘)에 올린 뒤
    #  URL 대신 "seoul+bike+sharing+demand.zip" 을 넣으세요
    raw.columns = ["date", "count", "hour", "temp", "humidity", "wind", "visibility",   # 위치 기반 리네임
                   "dewpoint", "solar", "rain", "snow", "season", "holiday", "functioning"]
    raw["datetime"] = (pd.to_datetime(raw["date"], format="%d/%m/%Y")   # 01/12/2017 = 2017년 12월 1일
                       + pd.to_timedelta(raw["hour"], unit="h"))
    df = raw.sort_values("datetime").set_index("datetime")
    df.index.freq = "h"                   # 등간격 1시간 — 168시간이 곧 7일이라는 뜻이 된다
    _s = df["count"].astype(float).mask(df["functioning"].eq("No"))   # 휴무는 0 이 아니라 관측 없음
    _y = _s.fillna(_s.shift(24 * 7))      # 1주일 전 같은 시각 값으로 채운다 (요일 효과를 지키려고)
    df["demand"] = _y.fillna(_y.shift(-24 * 7))   # 1주일 전도 휴무인 시각만 1주일 뒤 값으로
    df["is_imputed"] = df["functioning"].eq("No")

FIT_START, TRAIN_END, VAL_END = "2018-09-01 00:00", "2018-11-16 23:00", "2018-11-23 23:00"
VAL_START = pd.Timestamp(TRAIN_END) + pd.Timedelta(hours=1)
y_train = df["demand"].loc[FIT_START:TRAIN_END]     # 학습 구간 보정 수요 (대/시간)
y_valid = df["demand"].loc[VAL_START:VAL_END]       # 검증 구간 실측 (대/시간)
ORDER, SEASONAL_ORDER = (1, 0, 1), (1, 1, 1, 24)    # 미션 1 과 같은 차수: SARIMA(1, 0, 1)(1, 1, 1)₂₄

_res = SARIMAX(y_train, order=ORDER, seasonal_order=SEASONAL_ORDER, trend=None,
               enforce_stationarity=True, enforce_invertibility=True).fit(disp=False)
_f = _res.get_forecast(steps=len(y_valid))          # 미션 1 과 같이 168시간을 한 번에
_ci = _f.conf_int(alpha=0.05)                       # 95% 예측구간
fc = pd.DataFrame({"actual": np.asarray(y_valid, dtype=float),
                   "pred": np.asarray(_f.predicted_mean, dtype=float),
                   "lo95": np.asarray(_ci.iloc[:, 0], dtype=float),
                   "hi95": np.asarray(_ci.iloc[:, 1], dtype=float)}, index=y_valid.index)
fc["width"] = fc["hi95"] - fc["lo95"]               # 쓸 변수 1 — 168행 예측표 (구간 길이 width 열 포함)
resid = _res.resid.iloc[24:]                        # 쓸 변수 2 — 계절차분으로 z 가 없는 앞 24시간을 뗀 잔차
_theta = float(_res.params["ma.S.L24"])             # 계절 MA(계절 이동평균) 계수 Θ₁ — −1.00 이면 먼 시각 예측이 시각별 평균에 가까워진다
_hour_mean = y_train.groupby(y_train.index.hour).mean()        # 학습 11주의 시각별 평균
_gap = float(np.abs(fc["pred"].iloc[-24:].values - _hour_mean.loc[fc.index[-24:].hour].values).max())

_n_covered = int(((fc["actual"] >= fc["lo95"]) & (fc["actual"] <= fc["hi95"])).sum())
print(f"예측표 fc — {len(fc)}행 x {fc.shape[1]}열   "
      f"{fc.index.min():%m-%d %H시} ~ {fc.index.max():%m-%d %H시}")
print("이 예측표는 미션 1 과 같은 적합에서 나온 결과라, 미션 1 과 같은 값이 나옵니다.")
_w = fc["width"]
print(f"구간 길이 — 첫 시각 {_w.iloc[0]:,.1f} 대 → 첫날 끝 {_w.iloc[23]:,.1f} 대 → 둘째 날 끝 {_w.iloc[47]:,.1f} 대 "
      f"→ 마지막 시각 {_w.iloc[-1]:,.1f} 대 ({_w.iloc[-1] / _w.iloc[0]:.2f}배)")
print("구간은 첫날 안에 대부분 길어지고, 그 뒤로는 조금씩만 늡니다.")
print(f"계절 MA(계절 이동평균) 계수 Θ₁ 이 {_theta:.2f} 이라, 먼 시각의 예측이 학습 11주의 시각별 평균으로 돌아갑니다.")
print(f"마지막 날 예측과 그 시각별 평균의 차이는 시각마다 {np.ceil(_gap):.0f}대 안쪽입니다.")
print(f"95% 예측구간이 포함한 실측 — {_n_covered}/{len(fc)}시간 ({_n_covered / len(fc):.1%})")
print(f"잔차 {len(resid):,}개 (계절차분으로 첫 24시간이 빠진 수) · 표준편차 {resid.std():,.1f} 대")
print("검정 도구 : acorr_ljungbox(잔차, lags=[24], model_df=4) — 돌려주는 표의 lb_stat 과 lb_pvalue 두 열")
print("            귀무가설은 「시차 24까지 자기상관이 없다」 이고, model_df 4 는 계수 4개(φ₁ · θ₁ · Φ₁ · Θ₁)를 자유도에서 뺀다는 뜻")
print("적합에 10초 안팎 걸렸습니다. 아래 「PROMPT 셀」은 이 변수들만 쓰고 다시 적합하지 않습니다.")
print("쓸 수 있는 이름 : fc (168행 예측표 — actual · pred · lo95 · hi95 · width, width 가 구간 길이) · resid (잔차)"
      " · y_valid (검증 실측) · df (표 전체) · acorr_ljungbox · rmse(실측, 예측) · mae(실측, 예측)"
      " · datefmt(ax, 표기, 눈금 수)")

In [ ]:
# 과제 — 프롬프트   # ai: prompt
# PROMPT 를 쓰고 → 도우미에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶
PROMPT = """

"""
# --- 여기부터 AI 코드 ---

**브리핑 세 문장**: 아래 세 문장을 이 셀에 그대로 채워 제출합니다. 채점을 받기 전에 내 출력만 보고 먼저 채웁니다.

> **결론**: 확신으로 전달할 시각은 ____시간이고 나머지 ____시간은 보류입니다. 구분 기준값은 구간 길이 ____ 대로 정했고, 그 값을 정한 운영 근거는 ____ 입니다.
>
> **위험**: ____ (하한이 음수인 ____시간을 어떻게 적어 전달할지, 그리고 잔차 Ljung-Box 가 시차 24 에서 남긴 한계를 내 수치로 한 문장)
>
> **코드가 답을 만드는 방식**: ____ (등급 열을 어떤 열의 어떤 비교로 만들었는지, 그리고 확신과 보류의 경계인 세로선을 어느 시각에 그었는지 한 문장)

## 3. 오늘의 정리와 넷째 날 예고 — 원점을 옮겨 가며 다시 측정하기

## 오늘의 정리

| 미션 | 알아낸 것 | 대표 숫자 |
|---|---|---|
| 미션 1 · 168시간 예측 | 11월 16일 23시에서 168시간을 한 번에 예측했고, 95% 예측구간이 대부분의 시각에서 실측을 담았습니다 | 168시간 중 162시간 |
| 미션 1 · 오차 비교 | 같은 검증 168시간에서 SARIMA 모형의 오차가 계절 나이브 168 보다 컸습니다 | RMSE 386.08대 대 205.25대 |
| 미션 1 · 어제 차수와 비교 | AIC 는 내려갔는데 검증 오차는 커졌습니다. AIC 는 한 시간 뒤를, 검증 오차는 1～168시간 뒤를 내리 맞힌 결과입니다 | AIC 706.65 감소, RMSE 271.22 → 386.08대 |
| 과제 · 인수인계표 | 먼 시각일수록 구간이 길어져 앞쪽 시각만 확신으로 적을 수 있고, 하한이 음수인 시각이 남습니다 | 구간 길이 767대 → 1,629대(2.12배) |

**168시간은 채웠지만, 오차가 1주일 전 값을 그대로 쓰는 계절 나이브 168 보다 컸습니다.** 뒤의 날들이 넘어야 할 기준선은 계절 나이브 168 의 RMSE 205.25대입니다.

<img src="https://otexts.com/fpp3/fpp_files/figure-html/beeraccuracy-1.png" width="640" height="320"/>
<p align="center"><em>▲ 호주 분기 맥주 생산에 규칙 4개로 만든 예측과 실제값. 파랑이 평균, 초록이 나이브(마지막 값 그대로), 분홍이 계절 나이브, 주황이 드리프트(처음과 끝 값을 이은 직선을 늘린 것), 검정이 실제값이다. 계절성이 뚜렷한 계열에서는 분홍 계절 나이브만 실제값이 오르고 내리는 모양을 따라간다 (출처: Hyndman &amp; Athanasopoulos, fpp3 §5.8)</em></p>

오차가 난 곳은 수준입니다. 예측선은 8시와 18시에 많은 하루 모양을 따라갔지만, 하루 평균으로 보면 7일 모두 예측이 실측보다 높았습니다.

| 기간 | 시간당 평균(대) |
|---|---|
| 학습 11주(9월 1일～11월 16일) | 987 |
| 1주일 전(11월 10～16일) | 830 |
| 검증 주(11월 17～23일) 실측 | 694 |
| 검증 주 SARIMA 예측 | 966 |

먼 시각의 SARIMA 예측은 학습 11주의 시각별 평균으로 돌아갑니다. 23일 8시 예측 1,368대가 학습 11주의 8시 평균 1,378대와 거의 같습니다. 계절 나이브 168 은 이미 내려온 1주일 전 실측을 그대로 써서 실측에 더 가까웠습니다.

<details class="lms-extra"><summary>자세히 — Θ₁ = −1.00 이면 먼 시각의 예측이 평균으로 돌아가는 까닭</summary>

24시간 전 잔차에 곱하는 계수 $\Theta_1$ 이 −1.00 입니다. 24시간 전 실측에서 그때 빗나간 몫을 전부 빼면 24시간 전의 예측이 남고, 그 예측도 전날 예측에서 왔습니다. 그래서 학습 첫날까지 이어 보면 예측은 11주 동안 같은 시각 값의 평균에 가까워집니다. 16일 23시 뒤로는 16일의 계절차분한 값과 잔차가 주는 몫만 하루에 $\Phi_1$ = 0.29 배로 줄어듭니다.

8시 예측은 17일 1,446 → 18일 1,391 → 23일 1,368대로 8시 평균 1,378대 근처로 다가갑니다.

</details>

이 모형이 쓰지 않은 정보 가운데 오차를 줄일 만한 것은 요일과 기온 둘입니다. 두 정보를 입력으로 쓰는 모형은 다섯째 날에 다룹니다.

| 모형에 더할 정보 | 근거 숫자 |
|---|---|
| 요일 | 17일(토) 8시 예측 1,446대, 실측 530대. 토요일 8시 잔차는 3일 −332대, 10일 −378대로 같은 방향이고, 1주일 전 같은 시각 잔차와의 자기상관은 0.386 |
| 기온 | 검증 주 평균 4.91도로, 1주일 전 9.37도보다 4.5도 낮음 |

## 넷째 날 예고 — 원점 하나가 아니라 여러 회차로 측정하기

RMSE 2개는 원점 11월 16일 23시 하나에서 계산한 값이라, 두 숫자의 순위가 그 한 주의 사정일 수 있습니다. 넷째 날에는 학습 구간의 끝을 한 주씩 옮겨 가며 여러 회차로 다시 측정하고 평균으로 비교합니다.

<img src="https://otexts.com/fpp3/fpp_files/figure-html/cv1-1.png" width="640" height="300"/>
<p align="center"><em>▲ 한 행이 한 회차인 교차검증. 파란 점이 학습에 사용한 관측, 주황 점이 그 회차에서 채점하는 시각 1개, 회색 점이 아직 사용하지 않은 미래이고, 아래로 갈수록 학습 구간이 한 시각씩 길어진다 (출처: fpp3 §5.10)</em></p>

마무리로 완성본과 내 코드가 달랐던 부분 1곳과 거기서 배운 것 한 문장을 적어 팀 채널에 공유합니다.

<div style="background:#F2F2F2;color:#000000;padding:28px;border-radius:12px;border:1px solid #E6E6E6;margin-top:16px;border-left:4px solid #051C2C">
<h2 style="color:#000000;margin:0 0 8px 0">수고했습니다 — 168시간 인수인계표 완료</h2>
<div style="color:#656565">어제 고른 차수에 계절 항을 더해 168시간을 채우고, 그 오차가 계절 나이브 168 보다 큰 것까지 확인했습니다.<br/>넷째 날에는 이 두 숫자를 원점을 옮겨 가며 여러 회차로 다시 측정합니다.</div>
</div>

> **[문제] 잔차 자기상관 막대 9개와 Ljung-Box p 값을 함께 읽기**
>
> 오늘 차수의 잔차 1,824개(계절차분으로 첫 24시간이 빠진 수)에 Ljung-Box 를 시차 24 로 적용하면 통계량 277.46 · p 값 3.16e-47 이 나옵니다. 같은 잔차의 자기상관을 시차 36 까지 계산하면 경계선(우연으로 볼 수 있는 범위의 경계) ±0.046 밖 막대가 9개입니다. 두 결과로 내리는 결론으로 맞는 것은?
>
> 1. 막대 9개는 규칙 없는 오차(백색잡음)에서 기대하는 1.8개(36 × 0.05)의 5배이고 p 값도 0.05 보다 작으니, 남은 자기상관을 한계로 적는다
> 2. p 값 3.16e-47 이 0 에 가까우니 자기상관이 없다는 뜻이고, 막대 9개는 그림에서만 보이는 차이라 진단은 통과다
> 3. 경계선 ±0.046 은 잔차가 1,824개라 좁아진 값이라 막대 개수로는 판단할 수 없고, 시차 24 하나를 본 p 값만 근거가 된다

> **[문제] 계절 나이브 168 보다 오차가 컸는데도 이 표를 배차팀에 전달하는 근거**
>
> 오늘 SARIMA 모형은 검증 168시간에서 1주일 전 값을 그대로 내는 계절 나이브 168 보다 큰 오차를 받았습니다. 그런데도 SARIMA 모형의 예측과 95% 예측구간으로 만든 표를 배차팀에 함께 전달합니다. 그 판단의 근거로 맞는 것은?
>
> 1. 규칙의 오차가 작았던 것은 이번 168시간에만 생긴 우연이라, 다음 주에는 두 방법의 순위가 반대가 될 것이기 때문이다
> 2. SARIMA 모형의 AIC 가 더 낮았으니, 검증 구간 오차가 커도 배차에 쓸 숫자는 모형 쪽이 더 정확하기 때문이다
> 3. 배차 대수는 오차가 작은 계절 나이브 168 로 내고, 이 표의 등급은 확정할 시각을 가르는 근거로 쓰기 때문이다